<a href="https://colab.research.google.com/github/VoevodovaNA/Master_thesis/blob/main/RTSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##############################################################
# ЯЧЕЙКА 1: УСТАНОВКА
##############################################################
# !pip -q install ultralytics opencv-python-headless

import os, shutil, zipfile, glob
import numpy as np
from collections import Counter
from google.colab import drive, files
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("✅ Ячейка 1 готова.")


##############################################################
# ЯЧЕЙКА 2: GOOGLE DRIVE + РАСПАКОВКА ДАТАСЕТА
##############################################################
drive.mount('/content/drive')

WORK      = "/content/drive/MyDrive/VKR_Crosswalk"
MODEL_DIR = f"{WORK}/models"
METRICS   = f"{WORK}/metrics"
RESULTS   = f"{WORK}/results"

SIGNS_DIR = "/content/VKR_RoadSigns_YOLO"  # локально в Colab (быстрее)

for d in [MODEL_DIR, METRICS, RESULTS]:
    os.makedirs(d, exist_ok=True)

# ─── Распаковка датасета ───
# Загрузите VKR_RoadSigns_YOLO.zip на Drive в VKR_Crosswalk/
# или загрузите прямо сюда:

zip_candidates = [
    f"{WORK}/VKR_RoadSigns_YOLO.zip",
    "/content/VKR_RoadSigns_YOLO.zip",
    "/content/drive/MyDrive/VKR_RoadSigns_YOLO.zip",
]

zip_path = None
for zp in zip_candidates:
    if os.path.exists(zp):
        zip_path = zp
        break

if zip_path is None:
    print("⬆️  Загрузите VKR_RoadSigns_YOLO.zip:")
    uploaded = files.upload()
    zip_path = f"/content/{next(iter(uploaded))}"

print(f"📦 Распаковка: {zip_path}")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/")

# Обновляем путь в data.yaml
data_yaml = f"{SIGNS_DIR}/data.yaml"
with open(data_yaml, 'w') as f:
    f.write(f"""path: {SIGNS_DIR}
train: images/train
val: images/val
test: images/test

nc: 5

names:
  0: "ped_crossing"
  1: "ped_zebra_cross"
  2: "stop"
  3: "green_light"
  4: "red_light"
""")

# Проверка
for split in ['train', 'val', 'test']:
    n_img = len(glob.glob(f"{SIGNS_DIR}/images/{split}/*.jpg"))
    n_lbl = len(glob.glob(f"{SIGNS_DIR}/labels/{split}/*.txt"))
    print(f"   {split}: {n_img} изображений, {n_lbl} меток")

print(f"\n📄 data.yaml: {data_yaml}")
print("✅ Ячейка 2 готова.")


##############################################################
# ЯЧЕЙКА 3: ВИЗУАЛЬНАЯ ПРОВЕРКА ДАТАСЕТА
##############################################################
import cv2
import random

CLASS_NAMES = ["ped_crossing", "ped_zebra_cross", "stop", "green_light", "red_light"]
COLORS = [(0,0,255), (255,0,0), (0,165,255), (0,200,0), (200,0,200)]

img_dir = f"{SIGNS_DIR}/images/train"
lbl_dir = f"{SIGNS_DIR}/labels/train"

all_imgs = sorted(glob.glob(f"{img_dir}/*.jpg"))
random.seed(42)
samples = random.sample(all_imgs, min(9, len(all_imgs)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(lbl_dir, base + '.txt')

    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f.read().strip().split('\n'):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                c = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1,y1), (x2,y2), c, 3)
                cv2.putText(img, CLASS_NAMES[cls_id],
                            (x1, max(15, y1-10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, c, 2)

    axes[i].imshow(img)
    axes[i].axis('off')

plt.suptitle('Примеры из train-выборки (5 классов)', fontsize=14)
plt.tight_layout()
plt.savefig(f"{METRICS}/signs_dataset_samples.png", dpi=150)
plt.show()
print("✅ Ячейка 3 готова.")


##############################################################
# ЯЧЕЙКА 4: ОБУЧЕНИЕ YOLOv8
##############################################################
from ultralytics import YOLO

# ═══════════════════════════════════════════════════
# ГИПЕРПАРАМЕТРЫ
# ═══════════════════════════════════════════════════
BASE_MODEL = "yolov8n.pt"     # nano (быстро) или yolov8s.pt (точнее)
EPOCHS     = 80
BATCH      = 16               # уменьшить до 8 если мало VRAM
IMGSZ      = 640
PATIENCE   = 20               # ранняя остановка

print(f"🚀 Обучение YOLOv8")
print(f"   Модель:  {BASE_MODEL}")
print(f"   Эпох:    {EPOCHS}")
print(f"   Batch:   {BATCH}")
print(f"   ImgSize: {IMGSZ}")
print()

model = YOLO(BASE_MODEL)

results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    mosaic=1.0,
    flipud=0.0,          # знаки не переворачиваем вертикально
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5,
    translate=0.15,
    scale=0.4,
    project="/content/runs",
    name="signs_train",
    exist_ok=True,
    verbose=True,
)

print("✅ Обучение завершено!")


##############################################################
# ЯЧЕЙКА 5: СОХРАНЕНИЕ МОДЕЛИ + МЕТРИКИ
##############################################################

TRAIN_DIR = "/content/runs/signs_train"

# ─── Копируем лучшую модель ───
best_pt = f"{TRAIN_DIR}/weights/best.pt"
dst_pt  = f"{MODEL_DIR}/signs_best.pt"

if os.path.exists(best_pt):
    shutil.copy2(best_pt, dst_pt)
    print(f"✅ signs_best.pt → {dst_pt} ({os.path.getsize(dst_pt)/1e6:.1f} МБ)")
else:
    last_pt = f"{TRAIN_DIR}/weights/last.pt"
    if os.path.exists(last_pt):
        shutil.copy2(last_pt, f"{MODEL_DIR}/signs_last.pt")
        print(f"⚠️  best.pt не найден, скопирован last.pt")

# ─── Копируем графики обучения ───
for fig_name in ['results.png', 'confusion_matrix.png',
                 'F1_curve.png', 'PR_curve.png',
                 'P_curve.png', 'R_curve.png',
                 'confusion_matrix_normalized.png']:
    src = os.path.join(TRAIN_DIR, fig_name)
    if os.path.exists(src):
        shutil.copy2(src, f"{METRICS}/signs_{fig_name}")

print(f"📈 Графики → {METRICS}/signs_*")

# ─── Валидация ───
model_eval = YOLO(dst_pt)
val = model_eval.val(data=data_yaml, imgsz=IMGSZ, batch=BATCH)

print(f"\n{'═' * 60}")
print(f" МЕТРИКИ — Модель дорожных знаков")
print(f"{'═' * 60}")
print(f"\n  mAP@0.5:      {val.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val.box.map:.4f}")
print(f"  Precision:     {val.box.mp:.4f}")
print(f"  Recall:        {val.box.mr:.4f}")

print(f"\n  {'Класс':<20} {'AP@0.5':<10}")
print(f"  {'─' * 30}")
for i, name in enumerate(CLASS_NAMES):
    if i < len(val.box.ap50):
        print(f"  {name:<20} {val.box.ap50[i]:.4f}")

print("\n✅ Ячейка 5 готова.")


##############################################################
# ЯЧЕЙКА 6: ИНТЕГРАЦИЯ В ОСНОВНУЮ СИСТЕМУ
# ============================================================
# Пример кода для добавления в System.proc()
##############################################################

INTEGRATION_CODE = '''
# ═══════════════════════════════════════════════════════
# ДОБАВИТЬ В __init__ класса System:
# ═══════════════════════════════════════════════════════

signs_path = f"{MODEL_DIR}/signs_best.pt"
self.m3 = YOLO(signs_path) if os.path.exists(signs_path) else None
if self.m3:
    print("✅ Signs model loaded")

# ═══════════════════════════════════════════════════════
# ДОБАВИТЬ В proc() ПОСЛЕ детекции crosswalk (r2):
# ═══════════════════════════════════════════════════════

# Детекция дорожных знаков
if self.m3 is not None:
    r3 = self.m3(pf, verbose=False, conf=0.40)
    for b in r3[0].boxes:
        cls_name = self.m3.names[int(b.cls[0])]
        cf = float(b.conf[0])
        x1, y1, x2, y2 = map(int, b.xyxy[0])

        if cls_name in ("ped_crossing", "ped_zebra_cross"):
            # Знак пешеходного перехода → активируем зону
            self.crossing = True
            self.cross_timer = 0
            self.dc["sign_" + cls_name] += 1

        if cls_name in ("red_light", "green_light"):
            # Сигнал светофора → обновляем тип перехода
            if cls_name == "red_light":
                self.regulated = True
            self.dc["sign_" + cls_name] += 1

        # Рисуем знак на кадре
        b_scaled = [int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy)]
        cv2.rectangle(img,
                      (b_scaled[0], b_scaled[1]),
                      (b_scaled[2], b_scaled[3]),
                      (255, 200, 0), 3)
        cv2.putText(img,
                    f"SIGN:{cls_name} {cf:.2f}",
                    (b_scaled[0], max(0, b_scaled[1]-15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (255, 200, 0), 2)
'''

print(INTEGRATION_CODE)

# Проверяем все модели
print("═" * 55)
print(" СТАТУС МОДЕЛЕЙ ВКР")
print("═" * 55)
for name, path in [
    ("YOLOv8 COCO",    f"{MODEL_DIR}/yolov8n.pt"),
    ("Crosswalk",      f"{MODEL_DIR}/crosswalk_best.pt"),
    ("Road Signs",     f"{MODEL_DIR}/signs_best.pt"),
]:
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1e6
        print(f"  ✅ {name:<16} {sz:.1f} МБ")
    else:
        print(f"  ❌ {name:<16} не найдена")

print("\n✅ Всё готово!")

✅ Ячейка 1 готова.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⬆️  Загрузите VKR_RoadSigns_YOLO.zip:


Saving VKR_RoadSigns_YOLO.zip to VKR_RoadSigns_YOLO.zip
📦 Распаковка: /content/VKR_RoadSigns_YOLO.zip
   train: 369 изображений, 369 меток
   val: 120 изображений, 120 меток
   test: 10 изображений, 10 меток

📄 data.yaml: /content/VKR_RoadSigns_YOLO/data.yaml
✅ Ячейка 2 готова.
✅ Ячейка 3 готова.
🚀 Обучение YOLOv8
   Модель:  yolov8n.pt
   Эпох:    80
   Batch:   16
   ImgSize: 640

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/VKR_RoadSigns_YOLO/data.yaml, degrees=5, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscr

In [ ]:
##############################################################
# ЯЧЕЙКА 1: УСТАНОВКА ЗАВИСИМОСТЕЙ
##############################################################
# !pip -q install ultralytics opencv-python-headless dataset-tools

import os, shutil, json, glob, random
import numpy as np
from collections import Counter
from tqdm import tqdm
from google.colab import drive
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("✅ Ячейка 1 готова.")


✅ Ячейка 1 готова.


In [ ]:
##############################################################
# ЯЧЕЙКА 2: ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И НАСТРОЙКА ПУТЕЙ
##############################################################

import os
from google.colab import drive

# Если Drive уже подключён — не подключаем повторно
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("✅ Google Drive уже подключён")

# ═══════════════════════════════════════════════════
# Основная рабочая директория ВКР
# ═══════════════════════════════════════════════════

WORK         = "/content/drive/MyDrive/VKR_Crosswalk"
MODEL_DIR    = f"{WORK}/models"
RESULTS      = f"{WORK}/results"
METRICS      = f"{WORK}/metrics"

# ═══════════════════════════════════════════════════
# Директория для датасета дорожных знаков
# ═══════════════════════════════════════════════════

SIGNS_WORK   = "/content/drive/MyDrive/VKR_RoadSigns"
SIGNS_RAW    = f"{SIGNS_WORK}/rtsd_raw"
SIGNS_YOLO   = f"{SIGNS_WORK}/rtsd_raw/yolo"
SIGNS_MODELS = f"{SIGNS_WORK}/models"

for d in [
    WORK,
    MODEL_DIR,
    RESULTS,
    METRICS,
    SIGNS_WORK,
    SIGNS_RAW,
    SIGNS_YOLO,
    SIGNS_MODELS
]:
    os.makedirs(d, exist_ok=True)

# ═══════════════════════════════════════════════════
# Целевые классы RTSD
# ═══════════════════════════════════════════════════

TARGET_CLASSES = {
    "5_19_1": 0,   # Пешеходный переход
    "5_19_2": 1,   # Пешеходный переход, если есть в датасете
    "1_22":   2,   # Пешеходный переход, предупреждающий
    "1_23":   3,   # Дети
}

YOLO_NAMES = {
    0: "5_19_1",
    1: "5_19_2",
    2: "1_22",
    3: "1_23",
}

print(f"📍 Рабочая папка ВКР: {WORK}")
print(f"📍 Рабочая папка знаков: {SIGNS_WORK}")
print(f"📍 YOLO-датасет знаков: {SIGNS_YOLO}")
print(f"🎯 Целевые классы: {list(TARGET_CLASSES.keys())}")
print("✅ Ячейка 2 готова.")

✅ Google Drive уже подключён
📍 Рабочая папка ВКР: /content/drive/MyDrive/VKR_Crosswalk
📍 Рабочая папка знаков: /content/drive/MyDrive/VKR_RoadSigns
📍 YOLO-датасет знаков: /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo
🎯 Целевые классы: ['5_19_1', '5_19_2', '1_22', '1_23']
✅ Ячейка 2 готова.


In [ ]:
import os
import shutil
import glob

BASE = "/content/drive/MyDrive/VKR_RoadSigns"
SIGNS_RAW = f"{BASE}/rtsd_raw"

os.makedirs(SIGNS_RAW, exist_ok=True)

# ищем zip в основной папке VKR_RoadSigns
zips = glob.glob(f"{BASE}/*.zip")

print("Найденные zip в VKR_RoadSigns:")
for z in zips:
    print(z)

for z in zips:
    dst = f"{SIGNS_RAW}/{os.path.basename(z)}"
    print(f"Перемещаю: {z} -> {dst}")
    shutil.move(z, dst)

print("Готово. Теперь в rtsd_raw:")
print(os.listdir(SIGNS_RAW))

Найденные zip в VKR_RoadSigns:
Готово. Теперь в rtsd_raw:
['yolo']


In [ ]:
import os
import glob

SIGNS_RAW = "/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw"
YOLO_DATA = f"{SIGNS_RAW}/yolo"

print("YOLO_DATA:", YOLO_DATA)

print("images/train:", len(glob.glob(f"{YOLO_DATA}/images/train/*")))
print("images/val:", len(glob.glob(f"{YOLO_DATA}/images/val/*")))
print("labels/train:", len(glob.glob(f"{YOLO_DATA}/labels/train/*.txt")))
print("labels/val:", len(glob.glob(f"{YOLO_DATA}/labels/val/*.txt")))

print("\nФайлы в yolo:")
print(os.listdir(YOLO_DATA))

YOLO_DATA: /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo
images/train: 0
images/val: 0
labels/train: 0
labels/val: 0

Файлы в yolo:
[]


In [ ]:
DATA_YAML = "/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/data.yaml"

In [ ]:
import os
import glob

DATA_YAML = "/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/data.yaml"

fixed_yaml = """path: /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo
train: images/train
val: images/val

names:
  0: 5_19_1
  1: 5_19_2
  2: 1_22
  3: 1_23
"""

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(fixed_yaml)

print("data.yaml исправлен:")
print(fixed_yaml)

print("Проверка файлов:")
print("DATA_YAML exists:", os.path.exists(DATA_YAML))
print("images/train:", len(glob.glob("/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/images/train/*")))
print("images/val:", len(glob.glob("/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/images/val/*")))
print("labels/train:", len(glob.glob("/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/labels/train/*.txt")))
print("labels/val:", len(glob.glob("/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/labels/val/*.txt")))

data.yaml исправлен:
path: /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo
train: images/train
val: images/val

names:
  0: 5_19_1
  1: 5_19_2
  2: 1_22
  3: 1_23

Проверка файлов:
DATA_YAML exists: True
images/train: 0
images/val: 0
labels/train: 0
labels/val: 0


In [ ]:
from ultralytics import YOLO

DATA_YAML = "/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/data.yaml"

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name="road_signs_pedestrian_crossing",
    project="/content/drive/MyDrive/VKR_RoadSigns/runs"
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/VKR_RoadSigns/rtsd_raw/yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=road_signs_pedestrian_crossing-2, nbs=64, nms=False, opset=None, optimi

In [ ]:
import os

best_path = "/content/drive/MyDrive/VKR_RoadSigns/runs/road_signs_pedestrian_crossing/weights/best.pt"

print("best.pt exists:", os.path.exists(best_path))
print(best_path)

best.pt exists: False
/content/drive/MyDrive/VKR_RoadSigns/runs/road_signs_pedestrian_crossing/weights/best.pt


In [ ]:
import os
import glob
import shutil

# Где искать результаты обучения
SEARCH_DIR = "/content/drive/MyDrive/VKR_RoadSigns/runs"

# Куда положить модель для основного кода
DST_DIR = "/content/drive/MyDrive/VKR_Crosswalk/models"
DST_PATH = f"{DST_DIR}/signs_best.pt"

os.makedirs(DST_DIR, exist_ok=True)

# Ищем все best.pt внутри runs
best_files = glob.glob(f"{SEARCH_DIR}/**/weights/best.pt", recursive=True)

print("Найденные best.pt:")
for p in best_files:
    print(p)

if not best_files:
    raise FileNotFoundError(
        "best.pt не найден. Значит обучение ещё не завершилось "
        "или YOLO сохранил результаты в другую папку."
    )

# Берём самый свежий best.pt
latest_best = max(best_files, key=os.path.getmtime)

print("\nИспользую самый свежий best.pt:")
print(latest_best)

shutil.copy2(latest_best, DST_PATH)

print("\n✅ Модель дорожных знаков скопирована как:")
print(DST_PATH)
print("exists:", os.path.exists(DST_PATH))

Найденные best.pt:


FileNotFoundError: best.pt не найден. Значит обучение ещё не завершилось или YOLO сохранил результаты в другую папку.

In [ ]:
signs_path = f"{MODEL_DIR}/signs_best.pt"

In [ ]:
from ultralytics import YOLO
import os

signs_path = "/content/drive/MyDrive/VKR_Crosswalk/models/signs_best.pt"

print("signs_best.pt exists:", os.path.exists(signs_path))

model = YOLO(signs_path)
print("Road Signs classes:")
print(model.names)

signs_best.pt exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/VKR_Crosswalk/models/signs_best.pt'

In [ ]:
import os
import shutil

# путь к модели, которая получилась после обучения на 1000 RTSD изображениях
src = "/content/drive/MyDrive/VKR_RoadSigns/runs/road_signs_pedestrian_crossing/weights/best.pt"

# путь, который использует твой основной код обработки видео
dst_dir = "/content/drive/MyDrive/VKR_Crosswalk/models"
dst = f"{dst_dir}/signs_best.pt"

os.makedirs(dst_dir, exist_ok=True)

if not os.path.exists(src):
    raise FileNotFoundError(f"Не найден обученный best.pt: {src}")

shutil.copy2(src, dst)

print("✅ Модель дорожных знаков скопирована:")
print(dst)
print("exists:", os.path.exists(dst))

FileNotFoundError: Не найден обученный best.pt: /content/drive/MyDrive/VKR_RoadSigns/runs/road_signs_pedestrian_crossing/weights/best.pt

In [ ]:
# ============================================================
# ПОЛНОСТЬЮ САМОСТОЯТЕЛЬНАЯ ЯЧЕЙКА
# Видео ВСЕГДА загружается вручную
# Используются 3 модели:
# 1) YOLOv8 COCO
# 2) Crosswalk
# 3) Road Signs
# После обработки результат автоматически скачивается
# ============================================================

import os
import time
import cv2
import numpy as np
from collections import Counter

from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import files

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. ПУТИ
# ============================================================

WORK = "/content/drive/MyDrive/VKR_Crosswalk"
MODEL_DIR = f"{WORK}/models"
RESULTS = f"{WORK}/results"
VIDEO_DIR = f"{WORK}/videos"

os.makedirs(WORK, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

coco_path = f"{MODEL_DIR}/yolov8n.pt"
cw_path = f"{MODEL_DIR}/crosswalk_best.pt"
signs_path = f"{MODEL_DIR}/signs_best.pt"


# ============================================================
# 2. ЗАГРУЗКА ВИДЕО ВРУЧНУЮ
# ============================================================

print("Загрузите видеофайл вручную:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Видео не было загружено.")

video_name = next(iter(uploaded))

if not video_name.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
    raise ValueError("Нужно загрузить видео в формате .mp4, .avi, .mov или .mkv")

VIDEO_PATH = f"{VIDEO_DIR}/{video_name}"

with open(VIDEO_PATH, "wb") as f:
    f.write(uploaded[video_name])

print(f"Видео успешно загружено: {VIDEO_PATH}")


# ============================================================
# 3. НАСТРОЙКИ
# ============================================================

PROC_W, PROC_H = 1280, 720
CONF_THRESHOLD = 0.30

V_AUTO = 50 / 3.6
MU = 0.7


# ============================================================
# 4. ПРОВЕРКА НАЛИЧИЯ МОДЕЛЕЙ
# ============================================================

if not os.path.exists(coco_path):
    print(f"COCO model not found: {coco_path}")
    print("Будет использована стандартная загрузка YOLO('yolov8n.pt').")
    coco_path = "yolov8n.pt"

if not os.path.exists(cw_path):
    print(f"Crosswalk model not found: {cw_path}")
    print("Модель пешеходного перехода не будет использована.")
    cw_path = None

if not os.path.exists(signs_path):
    print(f"Road Signs model not found: {signs_path}")
    print("Модель дорожных знаков не будет использована.")
    signs_path = None


# ============================================================
# 5. ЗАГРУЗКА МОДЕЛЕЙ
# ============================================================

print("\nLoading models...")

m_coco = YOLO(coco_path)
print("✅ YOLOv8 COCO loaded")

if cw_path is not None:
    m_cw = YOLO(cw_path)
    print("✅ Crosswalk model loaded")
    print(f"Crosswalk classes: {m_cw.names}")
else:
    m_cw = None

if signs_path is not None:
    m_signs = YOLO(signs_path)
    print("✅ Road Signs model loaded")
    print(f"Road Signs classes: {m_signs.names}")
else:
    m_signs = None

print("\n═══════════════════════════════════════════════════════")
print("СТАТУС МОДЕЛЕЙ ВКР")
print("═══════════════════════════════════════════════════════")
print("  ✅ YOLOv8 COCO")
print("  ✅ Crosswalk" if m_cw is not None else "  ❌ Crosswalk не загружена")
print("  ✅ Road Signs" if m_signs is not None else "  ❌ Road Signs не загружена")
print("═══════════════════════════════════════════════════════\n")


# ============================================================
# 6. БЫСТРАЯ ПРОВЕРКА МОДЕЛИ ПЕРЕХОДА НА ОДНОМ КАДРЕ
# ============================================================

if m_cw is not None:
    print("\n=== ПРОВЕРКА КАЧЕСТВА МОДЕЛИ ПЕРЕХОДА ===")

    cap_test = cv2.VideoCapture(VIDEO_PATH)
    cap_test.set(cv2.CAP_PROP_POS_FRAMES, 50)
    ok_test, frame_test = cap_test.read()
    cap_test.release()

    if ok_test:
        frame_small = cv2.resize(frame_test, (PROC_W, PROC_H))

        results_015 = m_cw(frame_small, conf=0.15, verbose=False)

        print("\nDetections at conf=0.15:")
        if len(results_015[0].boxes) == 0:
            print("   No detections")
        else:
            for b in results_015[0].boxes:
                cls = m_cw.names[int(b.cls[0])]
                conf = float(b.conf[0])
                print(f"   {cls}: {conf:.3f}")

        results_025 = m_cw(frame_small, conf=0.25, verbose=False)

        print("\nDetections at conf=0.25:")
        if len(results_025[0].boxes) == 0:
            print("   No detections")
        else:
            for b in results_025[0].boxes:
                cls = m_cw.names[int(b.cls[0])]
                conf = float(b.conf[0])
                print(f"   {cls}: {conf:.3f}")
    else:
        print("Не удалось прочитать 50-й кадр видео для теста.")


# ============================================================
# 7. КАЛМАНОВСКИЙ ФИЛЬТР
# ============================================================

class KF:
    def __init__(self):
        self.F = np.eye(8)

        for i in range(4):
            self.F[i, 4 + i] = 1.0

        self.H = np.eye(4, 8)
        self.sp = 1.0 / 20
        self.sv = 1.0 / 160

    def init(self, m):
        mn = np.zeros(8)
        mn[:4] = m

        st = (
            [2 * self.sp * m[3]] * 2
            + [1e-2, 2 * self.sp * m[3]]
            + [10 * self.sv * m[3]] * 2
            + [1e-5, 10 * self.sv * m[3]]
        )

        return mn, np.diag(np.square(st))

    def pred(self, mn, cv):
        st = (
            [self.sp * mn[3]] * 2
            + [1e-2, self.sp * mn[3]]
            + [self.sv * mn[3]] * 2
            + [1e-5, self.sv * mn[3]]
        )

        mn_new = self.F @ mn
        cv_new = self.F @ cv @ self.F.T + np.diag(np.square(st))

        return mn_new, cv_new

    def upd(self, mn, cv, z):
        st = [self.sp * mn[3]] * 2 + [1e-2, self.sp * mn[3]]
        R = np.diag(np.square(st))

        S = self.H @ cv @ self.H.T + R
        K = cv @ self.H.T @ np.linalg.inv(S)

        mn_new = mn + K @ (z - self.H @ mn)
        cv_new = (np.eye(8) - K @ self.H) @ cv

        return mn_new, cv_new


# ============================================================
# 8. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ ТРЕКИНГА
# ============================================================

def iou_(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(1, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(1, (b[2] - b[0]) * (b[3] - b[1]))

    union = area_a + area_b - inter

    return inter / union


def to_z(b):
    w = b[2] - b[0]
    h = max(b[3] - b[1], 1)

    cx = b[0] + w / 2
    cy = b[1] + h / 2
    aspect = w / h

    return np.array([cx, cy, aspect, h])


def to_b(z):
    w = z[2] * z[3]

    x1 = z[0] - w / 2
    y1 = z[1] - z[3] / 2
    x2 = z[0] + w / 2
    y2 = z[1] + z[3] / 2

    return [x1, y1, x2, y2]


# ============================================================
# 9. КЛАСС ТРЕКА
# ============================================================

class Trk:
    _n = 1

    def __init__(self, bb, cl, cf):
        self.id = Trk._n
        Trk._n += 1

        self.cls = cl
        self.cf = cf

        self.kf = KF()
        self.mn, self.cv = self.kf.init(to_z(bb))

        self.hits = 1
        self.age = 0

        self.tr = [(self.mn[0], self.mn[1])]
        self.bbs = [bb]

    def predict(self):
        self.mn, self.cv = self.kf.pred(self.mn, self.cv)
        self.age += 1

    def update(self, bb, cl, cf):
        self.mn, self.cv = self.kf.upd(self.mn, self.cv, to_z(bb))

        self.hits += 1
        self.age = 0
        self.cls = cl
        self.cf = cf

        self.tr.append((self.mn[0], self.mn[1]))
        self.bbs.append(bb)

        if len(self.tr) > 90:
            self.tr.pop(0)

        if len(self.bbs) > 90:
            self.bbs.pop(0)

    def bbox(self):
        return to_b(self.mn[:4])

    def ok(self):
        return self.hits >= 2


# ============================================================
# 10. ТРЕКЕР
# ============================================================

class Tracker:
    def __init__(self):
        self.tracks = []

    def update(self, dets):
        for t in self.tracks:
            t.predict()

        if self.tracks and dets:
            C = np.zeros((len(self.tracks), len(dets)))

            for i, t in enumerate(self.tracks):
                for j, d in enumerate(dets):
                    C[i, j] = 1 - iou_(t.bbox(), d["bbox"])

            rs, cs = linear_sum_assignment(C)

            matched_dets = set()

            for r, c in zip(rs, cs):
                if C[r, c] < 0.75:
                    self.tracks[r].update(
                        dets[c]["bbox"],
                        dets[c]["class"],
                        dets[c]["conf"]
                    )
                    matched_dets.add(c)

            for j, d in enumerate(dets):
                if j not in matched_dets:
                    self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        elif dets:
            for d in dets:
                self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        self.tracks = [t for t in self.tracks if t.age <= 45]

        return [t for t in self.tracks if t.ok()]


# ============================================================
# 11. ОЦЕНКА РАССТОЯНИЯ
# ============================================================

HEIGHTS = {
    "person": 1.70,
    "bicycle": 1.10,
    "car": 1.50,
    "bus": 3.00,
    "truck": 3.50
}


def est_dist(bb, cl, focal=800):
    if cl not in HEIGHTS:
        return None

    h = bb[3] - bb[1]

    if h <= 10:
        return None

    d = (HEIGHTS[cl] * focal) / h

    if 0.5 < d < 80:
        return d

    return None


# ============================================================
# 12. TTC И РЕШЕНИЯ
# ============================================================

def s_stop():
    return V_AUTO * 0.2 + V_AUTO ** 2 / (2 * MU * 9.81)


def compute_ttc(d, v_app=0):
    vr = V_AUTO + v_app

    if vr > 0.3:
        return d / vr

    return 999.0


def v_approach(trk):
    if len(trk.bbs) < 4:
        return 0.0

    h1 = trk.bbs[-1][3] - trk.bbs[-1][1]
    h0 = trk.bbs[-4][3] - trk.bbs[-4][1]

    if h0 <= 0:
        return 0.0

    return max(0, (h1 - h0) / max(h0, 1) * 3.0)


# ============================================================
# 13. COCO КЛАССЫ И ФИЛЬТР СВЕТОФОРА
# ============================================================

COCO_TGT = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic light"
}


def valid_tl(bb):
    w = bb[2] - bb[0]
    h = bb[3] - bb[1]

    if w < 8 or h < 15:
        return False

    ratio = h / max(w, 1)

    if ratio < 1.5 or ratio > 5:
        return False

    if bb[1] > PROC_H * 0.6:
        return False

    return True


# ============================================================
# 14. КЛАССЫ ДОРОЖНЫХ ЗНАКОВ
# ============================================================

# Эти имена можно адаптировать под имена классов твоей модели signs_best.pt.
# Я добавил несколько вариантов, чтобы код работал и с названиями из RTSD,
# и с более простыми названиями типа ped_crossing.

PEDESTRIAN_SIGN_CLASSES = {
    "ped_crossing",
    "ped_zebra_cross",
    "5_19_1",
    "5_19_2",
    "5.19.1",
    "5.19.2",
    "1_22",
    "1.22",
    "crosswalk_sign",
    "pedestrian_crossing"
}

CHILDREN_SIGN_CLASSES = {
    "1_23",
    "1.23",
    "children",
    "children_sign",
    "school_zone"
}

TRAFFIC_LIGHT_SIGN_CLASSES = {
    "red_light",
    "green_light",
    "traffic_light",
    "traffic light"
}


def is_pedestrian_related_sign(cls_name):
    return cls_name in PEDESTRIAN_SIGN_CLASSES or cls_name in CHILDREN_SIGN_CLASSES


def is_traffic_light_related(cls_name):
    return cls_name in TRAFFIC_LIGHT_SIGN_CLASSES


# ============================================================
# 15. ИНИЦИАЛИЗАЦИЯ ВИДЕО
# ============================================================

tracker = Tracker()

dc = Counter()
timing = []
decisions_log = []
frame_decisions = []

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Не удалось открыть видео: {VIDEO_PATH}")

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if total <= 0:
    cap.release()

    cap = cv2.VideoCapture(VIDEO_PATH)
    total = 0

    while True:
        ok, _ = cap.read()
        if not ok:
            break
        total += 1

    cap.release()
    cap = cv2.VideoCapture(VIDEO_PATH)

base_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
out_path = f"{RESULTS}/{base_name}_result.mp4"

wr = cv2.VideoWriter(
    out_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

if not wr.isOpened():
    raise RuntimeError("Не удалось создать выходной видеофайл.")

print(f"\nVideo: {W}x{H}, {fps:.0f} fps, {total} frames")
print(f"Output path: {out_path}")


# ============================================================
# 16. ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ ВИДЕО
# ============================================================

crossing = False
cross_timer = 0
regulated = False
cw_bb = None

for idx in tqdm(range(total), desc="Processing"):
    ok, frame = cap.read()

    if not ok:
        break

    oh, ow = frame.shape[:2]

    pf = cv2.resize(frame, (PROC_W, PROC_H))

    sx = ow / PROC_W
    sy = oh / PROC_H

    t0 = time.time()

    dets = []
    sign_dets = []

    # --------------------------------------------------------
    # COCO DETECTION
    # --------------------------------------------------------

    t1 = time.time()

    r1 = m_coco(pf, verbose=False, conf=0.32)

    tc1 = (time.time() - t1) * 1000

    tl_found = False

    for b in r1[0].boxes:
        ci = int(b.cls[0])

        if ci not in COCO_TGT:
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0])
        cf = float(b.conf[0])
        nm = COCO_TGT[ci]

        if cf < 0.32:
            continue

        if nm == "traffic light":
            if cf >= 0.40 and valid_tl([x1, y1, x2, y2]):
                tl_found = True
                regulated = True
                dc["traffic_light_coco"] += 1
            continue

        dets.append({
            "bbox": [x1, y1, x2, y2],
            "class": nm,
            "conf": cf
        })

        dc[nm] += 1

    # --------------------------------------------------------
    # CROSSWALK DETECTION
    # --------------------------------------------------------

    t2 = time.time()
    tc2 = 0

    if m_cw is not None:
        r2 = m_cw(pf, verbose=False, conf=0.25)

        tc2 = (time.time() - t2) * 1000

        for b in r2[0].boxes:
            cf = float(b.conf[0])

            if cf >= 0.25:
                x1, y1, x2, y2 = map(int, b.xyxy[0])

                crossing = True
                cross_timer = 0
                cw_bb = [x1, y1, x2, y2]

                dc["crosswalk"] += 1

    # --------------------------------------------------------
    # ROAD SIGNS DETECTION
    # --------------------------------------------------------

    t_signs = time.time()
    tc_signs = 0

    if m_signs is not None:
        r3 = m_signs(pf, verbose=False, conf=0.40)
        tc_signs = (time.time() - t_signs) * 1000

        for b in r3[0].boxes:
            cls_name = m_signs.names[int(b.cls[0])]
            cf = float(b.conf[0])
            x1, y1, x2, y2 = map(int, b.xyxy[0])

            sign_dets.append({
                "bbox": [x1, y1, x2, y2],
                "class": cls_name,
                "conf": cf
            })

            if is_pedestrian_related_sign(cls_name):
                crossing = True
                cross_timer = 0
                dc["sign_" + cls_name] += 1

            if is_traffic_light_related(cls_name):
                regulated = True
                crossing = True
                cross_timer = 0
                dc["sign_" + cls_name] += 1

    # --------------------------------------------------------
    # ОБНОВЛЕНИЕ СОСТОЯНИЯ ЗОНЫ ПЕРЕХОДА
    # --------------------------------------------------------

    if tl_found and not crossing:
        crossing = True
        cross_timer = 0

    if crossing:
        cross_timer += 1

        if cross_timer > 150:
            crossing = False
            regulated = False
            cw_bb = None

    # --------------------------------------------------------
    # TRACKING
    # --------------------------------------------------------

    t3 = time.time()

    active = tracker.update(dets)

    tc3 = (time.time() - t3) * 1000

    # --------------------------------------------------------
    # DECISION LOGIC
    # --------------------------------------------------------

    t4 = time.time()

    per_track = []
    overall = "NO_ACTION"
    reason = "Crossing zone not detected"

    if crossing:
        persons = []

        for t in active:
            if t.cls != "person":
                continue

            d = est_dist(t.bbox(), "person")

            if d is None:
                continue

            va = v_approach(t)
            tc_val = compute_ttc(d, va)

            persons.append((t, d, tc_val))

        ss = s_stop()

        if not persons:
            overall = "SLOW_DOWN"
            reason = "Crossing or pedestrian-related sign detected"

        else:
            worst = 0

            for t, d, tc_val in persons:
                if d <= 10 or tc_val < 1.5 or d <= ss:
                    a = "EMERGENCY_BRAKE"
                    p = 3

                elif d <= 20 or tc_val < 2.5:
                    a = "SLOW_DOWN"
                    p = 2

                elif d <= 35 or tc_val < 4.0:
                    a = "WARNING"
                    p = 1

                else:
                    a = "NO_ACTION"
                    p = 0

                per_track.append((t.id, a, tc_val, d))

                if p > worst:
                    worst = p
                    overall = a

            closest = min(persons, key=lambda x: x[1])

            prefix = "[Regulated]" if regulated else "[Unregulated]"

            labels = {
                "EMERGENCY_BRAKE": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - EMERGENCY",
                "SLOW_DOWN": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - slow down",
                "WARNING": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - warning",
                "NO_ACTION": f"{prefix} Pedestrians at safe distance"
            }

            reason = labels.get(overall, "")

    tc4 = (time.time() - t4) * 1000

    # --------------------------------------------------------
    # DRAWING
    # --------------------------------------------------------

    t5 = time.time()

    img = frame.copy()

    def sb(b):
        return [
            int(b[0] * sx),
            int(b[1] * sy),
            int(b[2] * sx),
            int(b[3] * sy)
        ]

    # Crosswalk zone
    if cw_bb is not None and crossing:
        cb = sb(cw_bb)

        cv2.rectangle(
            img,
            (cb[0], cb[1]),
            (cb[2], cb[3]),
            (0, 255, 0),
            4
        )

        cv2.putText(
            img,
            "CROSSWALK",
            (cb[0], max(0, cb[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 255, 0),
            3
        )

    # Road signs
    for sdet in sign_dets:
        b = sb(sdet["bbox"])
        cls_name = sdet["class"]
        cf = sdet["conf"]

        if is_pedestrian_related_sign(cls_name):
            sign_color = (255, 200, 0)
        elif is_traffic_light_related(cls_name):
            sign_color = (0, 255, 255)
        else:
            sign_color = (180, 180, 180)

        cv2.rectangle(
            img,
            (b[0], b[1]),
            (b[2], b[3]),
            sign_color,
            3
        )

        cv2.putText(
            img,
            f"SIGN:{cls_name} {cf:.2f}",
            (b[0], max(0, b[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            sign_color,
            2
        )

    # Detections
    for d in dets:
        if d["conf"] < CONF_THRESHOLD:
            continue

        b = sb(d["bbox"])

        if d["class"] == "person":
            color = (0, 0, 255)
        else:
            color = (255, 165, 0)

        cv2.rectangle(
            img,
            (b[0], b[1]),
            (b[2], b[3]),
            color,
            3
        )

        lbl = f"{d['class']} {d['conf']:.2f}"

        dst = est_dist(d["bbox"], d["class"])

        if dst:
            lbl += f" {dst:.1f}m"

        cv2.putText(
            img,
            lbl,
            (b[0], max(0, b[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            color,
            2
        )

    # Tracks
    for t in active:
        if len(t.tr) > 2:
            pts = np.array(
                [(int(p[0] * sx), int(p[1] * sy)) for p in t.tr],
                np.int32
            )

            cv2.polylines(
                img,
                [pts],
                False,
                (0, 255, 255),
                4
            )

            lp = pts[-1]

            cv2.putText(
                img,
                f"ID:{t.id}",
                (lp[0] - 20, lp[1] - 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 255),
                3
            )

    # Top panel
    ov = img.copy()

    cv2.rectangle(
        ov,
        (0, 0),
        (ow, 220),
        (0, 0, 0),
        -1
    )

    cv2.addWeighted(
        ov,
        0.7,
        img,
        0.3,
        0,
        img
    )

    # Crossing type
    if crossing:
        ct = "REGULATED" if regulated else "UNREGULATED"
        cc = (0, 200, 255) if regulated else (0, 100, 255)
    else:
        ct = "NOT DETECTED"
        cc = (0, 200, 0)

    cv2.putText(
        img,
        f"CROSSING: {ct}",
        (20, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.3,
        cc,
        3
    )

    # Decision
    acm = {
        "EMERGENCY_BRAKE": ((0, 0, 255), "EMERGENCY BRAKE"),
        "SLOW_DOWN": ((0, 165, 255), "SLOW DOWN"),
        "WARNING": ((0, 255, 255), "WARNING"),
        "NO_ACTION": ((0, 200, 0), "NO ACTION")
    }

    ac, al = acm.get(overall, ((255, 255, 255), overall))

    cv2.putText(
        img,
        f"DECISION: {al}",
        (20, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.6,
        ac,
        4
    )

    cv2.putText(
        img,
        reason[:80],
        (20, 145),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (200, 200, 200),
        2
    )

    ss_val = s_stop()
    np_ = len([t for t in active if t.cls == "person"])

    cv2.putText(
        img,
        f"v={V_AUTO * 3.6:.0f}km/h  S_stop={ss_val:.1f}m  Pedestrians={np_}",
        (20, 185),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (180, 180, 180),
        2
    )

    # Per-pedestrian table
    xr = max(20, ow - 700)

    cv2.putText(
        img,
        "PEDESTRIANS:",
        (xr, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2
    )

    for i, (tid, a2, tc_v, d2) in enumerate(per_track[:5]):
        yy = 80 + i * 32

        c2, a2l = acm.get(a2, ((255, 255, 255), "?"))

        cv2.putText(
            img,
            f"ID:{tid} D={d2:.1f}m TTC={min(tc_v, 99):.1f}s {a2l}",
            (xr, yy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            c2,
            2
        )

    tc5 = (time.time() - t5) * 1000
    tt = (time.time() - t0) * 1000

    # Logs
    timing.append({
        "c1": tc1,
        "c2": tc2,
        "signs": tc_signs,
        "tr": tc3,
        "dm": tc4,
        "vis": tc5,
        "tot": tt
    })

    frame_decisions.append((idx, overall, reason))

    for item in per_track:
        decisions_log.append({
            "f": idx,
            "id": item[0],
            "act": item[1],
            "ttc": item[2],
            "dist": item[3]
        })

    wr.write(img)


cap.release()
wr.release()


# ============================================================
# 17. МЕТРИКИ И РЕЗУЛЬТАТЫ
# ============================================================

print(f"\n{'=' * 60}")
print("RESULTS")
print("=" * 60)

print("\nDetections:")

if len(dc) == 0:
    print("   No detections")
else:
    for c, n in dc.most_common():
        print(f"   {c:<25}: {n}")

if timing:
    print("\nTiming per frame:")

    for k, nm in [
        ("c1", "COCO det"),
        ("c2", "Crosswalk det"),
        ("signs", "Road signs det"),
        ("tr", "Tracking"),
        ("dm", "Decision"),
        ("vis", "Drawing"),
        ("tot", "TOTAL")
    ]:
        vals = [t[k] for t in timing]
        print(f"   {nm:<20}: {np.mean(vals):.1f} ms")

    avg_total = np.mean([t["tot"] for t in timing])

    if avg_total > 0:
        print(f"   FPS: {1000 / avg_total:.1f}")

if frame_decisions:
    print("\nDecisions summary:")

    acounter = Counter(f[1] for f in frame_decisions)
    tf = len(frame_decisions)

    for a, n in acounter.most_common():
        label = {
            "EMERGENCY_BRAKE": "EMERGENCY BRAKE",
            "SLOW_DOWN": "SLOW DOWN",
            "WARNING": "WARNING",
            "NO_ACTION": "NO ACTION"
        }.get(a, a)

        print(f"   {label:<25}: {n} frames ({n / tf * 100:.1f}%)")

print("\nBraking distance, v=50 km/h:")

for nm, mu in [
    ("Dry", 0.7),
    ("Wet", 0.5),
    ("Ice", 0.3)
]:
    v = V_AUTO

    driver_stop = v * 1.2 + v ** 2 / (2 * mu * 9.81)
    system_stop = v * 0.2 + v ** 2 / (2 * mu * 9.81)
    gain = driver_stop - system_stop

    print(
        f"   {nm}: "
        f"driver={driver_stop:.1f}m, "
        f"system={system_stop:.1f}m, "
        f"gain={gain:.1f}m"
    )


# ============================================================
# 18. АВТОМАТИЧЕСКОЕ СКАЧИВАНИЕ РЕЗУЛЬТАТА
# ============================================================

print(f"\nDone! Video saved to: {out_path}")

if os.path.exists(out_path):
    print("Начинается автоматическое скачивание видео...")
    time.sleep(1)
    files.download(out_path)
else:
    print(f"Файл результата не найден: {out_path}")

Загрузите видеофайл вручную:


Saving IMG_7721.MP4 to IMG_7721.MP4
Видео успешно загружено: /content/drive/MyDrive/VKR_Crosswalk/videos/IMG_7721.MP4

Loading models...
✅ YOLOv8 COCO loaded
✅ Crosswalk model loaded
Crosswalk classes: {0: 'crosswalk'}
✅ Road Signs model loaded
Road Signs classes: {0: '5191', 1: '5192', 2: '122', 3: '123'}

═══════════════════════════════════════════════════════
СТАТУС МОДЕЛЕЙ ВКР
═══════════════════════════════════════════════════════
  ✅ YOLOv8 COCO
  ✅ Crosswalk
  ✅ Road Signs
═══════════════════════════════════════════════════════


=== ПРОВЕРКА КАЧЕСТВА МОДЕЛИ ПЕРЕХОДА ===

Detections at conf=0.15:
   No detections

Detections at conf=0.25:
   No detections

Video: 1280x576, 28 fps, 877 frames
Output path: /content/drive/MyDrive/VKR_Crosswalk/results/IMG_7721_result.mp4


Processing: 100%|██████████| 877/877 [00:43<00:00, 20.08it/s]



RESULTS

Detections:
   car                      : 4631
   person                   : 2098
   bus                      : 177
   truck                    : 171
   traffic_light_coco       : 88

Timing per frame:
   COCO det            : 10.7 ms
   Crosswalk det       : 8.8 ms
   Road signs det      : 9.1 ms
   Tracking            : 2.0 ms
   Decision            : 0.0 ms
   Drawing             : 5.1 ms
   TOTAL               : 37.1 ms
   FPS: 26.9

Decisions summary:
   NO ACTION                : 427 frames (48.7%)
   EMERGENCY BRAKE          : 388 frames (44.2%)
   SLOW DOWN                : 62 frames (7.1%)

Braking distance, v=50 km/h:
   Dry: driver=30.7m, system=16.8m, gain=13.9m
   Wet: driver=36.3m, system=22.4m, gain=13.9m
   Ice: driver=49.4m, system=35.6m, gain=13.9m

Done! Video saved to: /content/drive/MyDrive/VKR_Crosswalk/results/IMG_7721_result.mp4
Начинается автоматическое скачивание видео...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:



##############################################################
# ЯЧЕЙКА 3: ЗАГРУЗКА RTSD
# ============================================================
# Вариант A: dataset-tools (Dataset Ninja / Supervisely)
# Вариант B: Kaggle
# Вариант C: Ручная загрузка
# ============================================================
# ВЫБЕРИТЕ ОДИН ВАРИАНТ, остальные закомментируйте.
##############################################################

# ─── ВАРИАНТ C: Ручная загрузка (РЕКОМЕНДУЕТСЯ) ───
# 1. Скачайте RTSD с: https://graphics.cs.msu.ru/projects/traffic-sign-recognition.html
#    или с Яндекс.Диска: https://yadi.sk/d/TX5k2hkEm9wqZ
# 2. Загрузите архив на Google Drive в папку:
#    VKR_RoadSigns/rtsd_raw/
# 3. Если это архив — распакуйте:

import zipfile

archives = glob.glob(f"{SIGNS_RAW}/*.zip")
for arch in archives:
    print(f"📦 Распаковка: {os.path.basename(arch)}")
    with zipfile.ZipFile(arch, 'r') as z:
        z.extractall(SIGNS_RAW)
    print(f"   ✅ Распакован")

# Поиск изображений и аннотаций
all_images = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
    all_images.extend(glob.glob(f"{SIGNS_RAW}/**/{ext}", recursive=True))

all_annotations = []
for ext in ['*.json', '*.csv', '*.txt', '*.xml']:
    all_annotations.extend(glob.glob(f"{SIGNS_RAW}/**/{ext}", recursive=True))

print(f"\n📊 Найдено в {SIGNS_RAW}:")
print(f"   Изображений: {len(all_images)}")
print(f"   Файлов аннотаций: {len(all_annotations)}")

# Показать структуру
for root, dirs, files_list in os.walk(SIGNS_RAW):
    depth = root.replace(SIGNS_RAW, '').count(os.sep)
    if depth > 2:
        continue
    indent = '   ' * depth
    print(f"{indent}📁 {os.path.basename(root)}/  ({len(files_list)} файлов)")

print("✅ Ячейка 3 готова.")


📦 Распаковка: RTSD_1000_pedestrian_signs.zip
   ✅ Распакован

📊 Найдено в /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw:
   Изображений: 1000
   Файлов аннотаций: 1000
📁 rtsd_raw/  (1 файлов)
   📁 yolo/  (1 файлов)
      📁 labels/  (0 файлов)
      📁 images/  (0 файлов)
✅ Ячейка 3 готова.


In [ ]:

##############################################################
# ЯЧЕЙКА 4: ПАРСИНГ АННОТАЦИЙ RTSD
# ============================================================
# Автоматически определяет формат аннотаций:
# - Supervisely JSON (от Dataset Ninja)
# - CSV/TSV (оригинальный RTSD формат)
# - YOLO txt (если уже конвертирован)
# - Pascal VOC XML
# ============================================================
##############################################################

import csv
import xml.etree.ElementTree as ET


def parse_supervisely_json(json_path, img_dir):
    """Парсинг формата Supervisely (Dataset Ninja)."""
    detections = []
    with open(json_path, 'r') as f:
        data = json.load(f)

    img_h = data.get('size', {}).get('height', 1)
    img_w = data.get('size', {}).get('width', 1)

    for obj in data.get('objects', []):
        cls_name = obj.get('classTitle', '')
        if cls_name not in TARGET_CLASSES:
            continue

        pts = obj.get('points', {})
        exterior = pts.get('exterior', [])
        if len(exterior) >= 2:
            x1, y1 = exterior[0]
            x2, y2 = exterior[1]

            # Нормализуем в YOLO формат
            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w = abs(x2 - x1) / img_w
            h = abs(y2 - y1) / img_h

            detections.append({
                'class': cls_name,
                'class_id': TARGET_CLASSES[cls_name],
                'cx': cx, 'cy': cy, 'w': w, 'h': h,
                'img_w': img_w, 'img_h': img_h
            })

    return detections


def parse_csv_annotations(csv_path):
    """
    Парсинг CSV-аннотаций RTSD.

    Ожидаемые форматы:
    1) filename, x_from, y_from, width, height, sign_class
    2) filename, x1, y1, x2, y2, sign_class
    3) или подобные вариации
    """
    detections_by_image = {}

    with open(csv_path, 'r') as f:
        # Пробуем определить разделитель
        sample = f.read(4096)
        f.seek(0)

        if '\t' in sample:
            delimiter = '\t'
        elif ';' in sample:
            delimiter = ';'
        else:
            delimiter = ','

        reader = csv.reader(f, delimiter=delimiter)

        # Определяем заголовок
        header = None
        first_row = next(reader, None)
        if first_row is None:
            return detections_by_image

        # Проверяем, является ли первая строка заголовком
        if any(h.lower() in ['filename', 'file', 'image', 'path',
                              'sign_class', 'class', 'sign']
               for h in first_row):
            header = [h.strip().lower() for h in first_row]
            rows = list(reader)
        else:
            header = None
            rows = [first_row] + list(reader)

        for row in rows:
            if len(row) < 5:
                continue

            try:
                if header:
                    row_dict = dict(zip(header, row))
                    filename = (row_dict.get('filename') or
                                row_dict.get('file') or
                                row_dict.get('image') or
                                row_dict.get('path', ''))
                    sign_class = (row_dict.get('sign_class') or
                                  row_dict.get('class') or
                                  row_dict.get('sign') or
                                  row_dict.get('label', ''))

                    # Координаты
                    if 'width' in row_dict and 'height' in row_dict:
                        x_from = float(row_dict.get('x_from', row_dict.get('x', 0)))
                        y_from = float(row_dict.get('y_from', row_dict.get('y', 0)))
                        w = float(row_dict['width'])
                        h = float(row_dict['height'])
                        bbox = ('xywh', x_from, y_from, w, h)
                    else:
                        x1 = float(row_dict.get('x1', row_dict.get('x_from', 0)))
                        y1 = float(row_dict.get('y1', row_dict.get('y_from', 0)))
                        x2 = float(row_dict.get('x2', row_dict.get('x_to', 0)))
                        y2 = float(row_dict.get('y2', row_dict.get('y_to', 0)))
                        bbox = ('xyxy', x1, y1, x2, y2)
                else:
                    # Без заголовка: пытаемся угадать формат
                    filename = row[0].strip()
                    sign_class = row[-1].strip()
                    coords = [float(x) for x in row[1:-1]]

                    if len(coords) == 4:
                        # Проверяем: xywh или xyxy
                        if coords[2] > coords[0] and coords[3] > coords[1]:
                            bbox = ('xyxy', *coords)
                        else:
                            bbox = ('xywh', *coords)
                    else:
                        continue

                sign_class = sign_class.strip()
                if sign_class not in TARGET_CLASSES:
                    continue

                if filename not in detections_by_image:
                    detections_by_image[filename] = []

                detections_by_image[filename].append({
                    'class': sign_class,
                    'class_id': TARGET_CLASSES[sign_class],
                    'bbox': bbox
                })

            except (ValueError, KeyError):
                continue

    return detections_by_image


def parse_voc_xml(xml_path):
    """Парсинг Pascal VOC XML."""
    detections = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find('size')
        img_w = int(size.find('width').text) if size is not None else 1
        img_h = int(size.find('height').text) if size is not None else 1

        for obj in root.findall('object'):
            name = obj.find('name').text.strip()
            if name not in TARGET_CLASSES:
                continue

            bndbox = obj.find('bndbox')
            x1 = float(bndbox.find('xmin').text)
            y1 = float(bndbox.find('ymin').text)
            x2 = float(bndbox.find('xmax').text)
            y2 = float(bndbox.find('ymax').text)

            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h

            detections.append({
                'class': name,
                'class_id': TARGET_CLASSES[name],
                'cx': cx, 'cy': cy, 'w': w, 'h': h,
                'img_w': img_w, 'img_h': img_h
            })
    except ET.ParseError:
        pass

    return detections


# ═══════════════════════════════════════════════════
# АВТООПРЕДЕЛЕНИЕ ФОРМАТА
# ═══════════════════════════════════════════════════

print("🔍 Определяю формат аннотаций RTSD...\n")

anno_format = None
parsed_data = {}  # {image_path: [(class_id, cx, cy, w, h), ...]}

# 1. Проверяем наличие Supervisely JSON
sly_jsons = glob.glob(f"{SIGNS_RAW}/**/ann/*.json", recursive=True)
if not sly_jsons:
    sly_jsons = glob.glob(f"{SIGNS_RAW}/**/*.json", recursive=True)
    # Фильтруем: только JSON с полем 'objects' — Supervisely формат
    real_sly = []
    for jp in sly_jsons[:20]:
        try:
            with open(jp, 'r') as f:
                d = json.load(f)
            if 'objects' in d and 'size' in d:
                real_sly.append(jp)
        except:
            pass
    sly_jsons = real_sly if real_sly else []

# 2. Проверяем CSV
csv_files = glob.glob(f"{SIGNS_RAW}/**/*.csv", recursive=True)
csv_files += glob.glob(f"{SIGNS_RAW}/**/*.tsv", recursive=True)

# 3. Проверяем XML (Pascal VOC)
xml_files = glob.glob(f"{SIGNS_RAW}/**/*.xml", recursive=True)

# 4. Проверяем YOLO txt
txt_labels = glob.glob(f"{SIGNS_RAW}/**/labels/**/*.txt", recursive=True)

if sly_jsons:
    anno_format = "supervisely"
    print(f"📋 Формат: Supervisely JSON ({len(sly_jsons)} файлов)")
elif csv_files:
    anno_format = "csv"
    print(f"📋 Формат: CSV/TSV ({len(csv_files)} файлов)")
elif xml_files:
    anno_format = "xml"
    print(f"📋 Формат: Pascal VOC XML ({len(xml_files)} файлов)")
elif txt_labels:
    anno_format = "yolo"
    print(f"📋 Формат: YOLO TXT (уже конвертирован, {len(txt_labels)} файлов)")
else:
    print("⚠️  Формат не определён!")
    print("   Проверьте содержимое папки:")
    print(f"   {SIGNS_RAW}")
    print("\n   Ожидаемая структура RTSD:")
    print("   rtsd_raw/")
    print("   ├── train/")
    print("   │   ├── images/  (или img/)")
    print("   │   └── annotations/  (CSV, JSON или XML)")
    print("   └── test/")
    print("       ├── images/")
    print("       └── annotations/")

print(f"\n🎯 Фильтруем классы: {list(TARGET_CLASSES.keys())}")
print("✅ Ячейка 4 готова.")


🔍 Определяю формат аннотаций RTSD...

⚠️  Формат не определён!
   Проверьте содержимое папки:
   /content/drive/MyDrive/VKR_RoadSigns/rtsd_raw

   Ожидаемая структура RTSD:
   rtsd_raw/
   ├── train/
   │   ├── images/  (или img/)
   │   └── annotations/  (CSV, JSON или XML)
   └── test/
       ├── images/
       └── annotations/

🎯 Фильтруем классы: ['5_19_1', '5_19_2', '1_22', '1_23']
✅ Ячейка 4 готова.


In [ ]:

##############################################################
# ЯЧЕЙКА 5: КОНВЕРТАЦИЯ В YOLO ФОРМАТ
# ============================================================
# Фильтрация + переиндексация + создание структуры
##############################################################

import cv2
from PIL import Image

# ═══════════════════════════════════════════════════
# Создаём структуру YOLO-датасета
# ═══════════════════════════════════════════════════
for split in ['train', 'val']:
    os.makedirs(f"{SIGNS_YOLO}/images/{split}", exist_ok=True)
    os.makedirs(f"{SIGNS_YOLO}/labels/{split}", exist_ok=True)

stats = Counter()
image_label_pairs = []  # [(img_src, labels_list), ...]

# ═══════════════════════════════════════════════════
# ПАРСИНГ В ЗАВИСИМОСТИ ОТ ФОРМАТА
# ═══════════════════════════════════════════════════

if anno_format == "supervisely":
    # Ищем пары: ann/*.json <-> img/*.*
    for json_path in tqdm(sly_jsons, desc="Парсинг Supervisely"):
        dets = parse_supervisely_json(json_path, os.path.dirname(json_path))
        if not dets:
            continue

        # Находим соответствующее изображение
        base = os.path.splitext(os.path.basename(json_path))[0]
        ann_dir = os.path.dirname(json_path)

        # Supervisely: ann/filename.json → img/filename.ext
        img_dir = ann_dir.replace('/ann', '/img')
        if not os.path.isdir(img_dir):
            img_dir = ann_dir.replace('\\ann', '\\img')

        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
            candidate = os.path.join(img_dir, base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break

        # Если не нашли в img/, ищем рядом
        if img_path is None:
            parent = os.path.dirname(ann_dir)
            for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
                candidate = os.path.join(parent, base + ext)
                if os.path.exists(candidate):
                    img_path = candidate
                    break

        if img_path is None:
            continue

        labels = []
        for d in dets:
            labels.append(f"{d['class_id']} {d['cx']:.6f} {d['cy']:.6f} {d['w']:.6f} {d['h']:.6f}")
            stats[d['class']] += 1

        image_label_pairs.append((img_path, labels))


elif anno_format == "csv":
    for csv_path in tqdm(csv_files, desc="Парсинг CSV"):
        dets_by_img = parse_csv_annotations(csv_path)
        csv_dir = os.path.dirname(csv_path)

        for filename, dets in dets_by_img.items():
            # Ищем изображение
            img_path = None
            candidates = [
                os.path.join(csv_dir, filename),
                os.path.join(csv_dir, 'images', filename),
                os.path.join(csv_dir, 'img', filename),
                os.path.join(csv_dir, '..', 'images', filename),
                os.path.join(csv_dir, '..', 'img', filename),
            ]
            # Ищем рекурсивно
            found = glob.glob(f"{SIGNS_RAW}/**/{filename}", recursive=True)
            candidates.extend(found)

            for c in candidates:
                if os.path.exists(c):
                    img_path = os.path.abspath(c)
                    break

            if img_path is None:
                continue

            # Получаем размер изображения для нормализации
            try:
                img = Image.open(img_path)
                img_w, img_h = img.size
            except:
                continue

            labels = []
            for d in dets:
                bbox_type = d['bbox'][0]
                if bbox_type == 'xywh':
                    _, x, y, w, h = d['bbox']
                    cx = (x + w / 2) / img_w
                    cy = (y + h / 2) / img_h
                    nw = w / img_w
                    nh = h / img_h
                elif bbox_type == 'xyxy':
                    _, x1, y1, x2, y2 = d['bbox']
                    cx = ((x1 + x2) / 2) / img_w
                    cy = ((y1 + y2) / 2) / img_h
                    nw = abs(x2 - x1) / img_w
                    nh = abs(y2 - y1) / img_h
                else:
                    continue

                labels.append(f"{d['class_id']} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                stats[d['class']] += 1

            if labels:
                image_label_pairs.append((img_path, labels))


elif anno_format == "xml":
    for xml_path in tqdm(xml_files, desc="Парсинг VOC XML"):
        dets = parse_voc_xml(xml_path)
        if not dets:
            continue

        base = os.path.splitext(os.path.basename(xml_path))[0]
        xml_dir = os.path.dirname(xml_path)

        img_path = None
        for search_dir in [xml_dir,
                           xml_dir.replace('/annotations', '/images'),
                           xml_dir.replace('/Annotations', '/JPEGImages'),
                           os.path.join(xml_dir, '..', 'images'),
                           os.path.join(xml_dir, '..', 'JPEGImages')]:
            for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
                candidate = os.path.join(search_dir, base + ext)
                if os.path.exists(candidate):
                    img_path = candidate
                    break
            if img_path:
                break

        if img_path is None:
            continue

        labels = []
        for d in dets:
            labels.append(f"{d['class_id']} {d['cx']:.6f} {d['cy']:.6f} {d['w']:.6f} {d['h']:.6f}")
            stats[d['class']] += 1

        image_label_pairs.append((img_path, labels))


elif anno_format == "yolo":
    # Уже в YOLO формате — нужно только переиндексировать

    # Ищем data.yaml чтобы узнать маппинг старых классов
    yaml_files = glob.glob(f"{SIGNS_RAW}/**/*.yaml", recursive=True)
    old_class_map = {}  # old_id -> class_name

    for yf in yaml_files:
        try:
            import yaml
            with open(yf, 'r') as f:
                yd = yaml.safe_load(f)
            if 'names' in yd:
                if isinstance(yd['names'], dict):
                    old_class_map = {int(k): v for k, v in yd['names'].items()}
                elif isinstance(yd['names'], list):
                    old_class_map = {i: v for i, v in enumerate(yd['names'])}
                print(f"   Найден data.yaml с {len(old_class_map)} классами")
                break
        except:
            continue

    for txt_path in tqdm(txt_labels, desc="Фильтрация YOLO"):
        base = os.path.splitext(os.path.basename(txt_path))[0]

        # Ищем изображение
        img_path = None
        label_dir = os.path.dirname(txt_path)
        img_dir = label_dir.replace('/labels/', '/images/').replace('\\labels\\', '\\images\\')

        for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
            candidate = os.path.join(img_dir, base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            continue

        with open(txt_path, 'r') as f:
            lines = f.readlines()

        labels = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            old_id = int(parts[0])
            if old_class_map:
                old_name = old_class_map.get(old_id, '')
            else:
                old_name = str(old_id)

            if old_name in TARGET_CLASSES:
                new_id = TARGET_CLASSES[old_name]
                labels.append(f"{new_id} {parts[1]} {parts[2]} {parts[3]} {parts[4]}")
                stats[old_name] += 1

        if labels:
            image_label_pairs.append((img_path, labels))

# ═══════════════════════════════════════════════════
# РАЗБИВКА TRAIN / VAL (80% / 20%)
# ═══════════════════════════════════════════════════

random.seed(42)
random.shuffle(image_label_pairs)

n_total = len(image_label_pairs)
n_train = int(n_total * 0.8)

train_pairs = image_label_pairs[:n_train]
val_pairs   = image_label_pairs[n_train:]

print(f"\n{'═' * 50}")
print(f"📊 СТАТИСТИКА ФИЛЬТРАЦИИ RTSD")
print(f"{'═' * 50}")
print(f"   Всего изображений с целевыми знаками: {n_total}")
print(f"   Train: {len(train_pairs)}")
print(f"   Val:   {len(val_pairs)}")
print(f"\n   Распределение по классам:")
for cls_name, count in stats.most_common():
    print(f"      {cls_name}: {count} аннотаций")

# ═══════════════════════════════════════════════════
# КОПИРОВАНИЕ ФАЙЛОВ В YOLO-СТРУКТУРУ
# ═══════════════════════════════════════════════════

def copy_pairs(pairs, split):
    """Копируем изображения и создаём label-файлы."""
    copied = 0
    for img_path, labels in tqdm(pairs, desc=f"Копирование {split}"):
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]

        # Уникальное имя (на случай дубликатов)
        dst_img = f"{SIGNS_YOLO}/images/{split}/{base}{ext}"
        dst_lbl = f"{SIGNS_YOLO}/labels/{split}/{base}.txt"

        # Если уже есть файл с таким именем — добавляем суффикс
        counter = 0
        while os.path.exists(dst_img):
            counter += 1
            dst_img = f"{SIGNS_YOLO}/images/{split}/{base}_{counter}{ext}"
            dst_lbl = f"{SIGNS_YOLO}/labels/{split}/{base}_{counter}.txt"

        shutil.copy2(img_path, dst_img)

        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(labels) + '\n')

        copied += 1
    return copied

n_tr = copy_pairs(train_pairs, 'train')
n_vl = copy_pairs(val_pairs, 'val')

print(f"\n✅ Скопировано: train={n_tr}, val={n_vl}")

# ═══════════════════════════════════════════════════
# СОЗДАНИЕ data.yaml
# ═══════════════════════════════════════════════════

data_yaml_content = f"""path: {SIGNS_YOLO}
train: images/train
val: images/val

nc: {len(TARGET_CLASSES)}

names:
  0: "5_19_1"
  1: "5_19_2"
  2: "1_22"
  3: "1_23"
"""

data_yaml_path = f"{SIGNS_YOLO}/data.yaml"

with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"\n📄 data.yaml создан: {data_yaml_path}")
print(data_yaml_content)
print("✅ Ячейка 5 готова.")


══════════════════════════════════════════════════
📊 СТАТИСТИКА ФИЛЬТРАЦИИ RTSD
══════════════════════════════════════════════════
   Всего изображений с целевыми знаками: 0
   Train: 0
   Val:   0

   Распределение по классам:


Копирование train: 0it [00:00, ?it/s]
Копирование val: 0it [00:00, ?it/s]


✅ Скопировано: train=0, val=0



📄 data.yaml создан: /content/drive/MyDrive/VKR_RoadSigns/dataset_yolo/data.yaml
path: /content/drive/MyDrive/VKR_RoadSigns/dataset_yolo
train: images/train
val: images/val

nc: 4

names:
  0: "5_19_1"
  1: "5_19_2"
  2: "1_22"
  3: "1_23"

✅ Ячейка 5 готова.


In [ ]:

##############################################################
# ЯЧЕЙКА 6: ПРОВЕРКА ДАТАСЕТА (ВИЗУАЛИЗАЦИЯ)
##############################################################

def visualize_samples(split='train', n=6):
    """Показать несколько примеров из датасета."""
    img_dir = f"{SIGNS_YOLO}/images/{split}"
    lbl_dir = f"{SIGNS_YOLO}/labels/{split}"

    images = glob.glob(f"{img_dir}/*.*")
    if not images:
        print(f"⚠️  Нет изображений в {split}")
        return

    samples = random.sample(images, min(n, len(images)))
    cols = 3
    rows = (len(samples) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes = np.array(axes).flatten()

    colors = {0: (0, 0, 255), 1: (255, 0, 0), 2: (0, 165, 255), 3: (0, 255, 0)}
    class_labels = {0: '5_19_1', 1: '5_19_2', 2: '1_22', 3: '1_23'}

    for i, img_path in enumerate(samples):
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w = img.shape[:2]

        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, base + '.txt')

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = [float(x) for x in parts[1:5]]

                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)

                    color = colors.get(cls_id, (255, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    label = class_labels.get(cls_id, str(cls_id))
                    cv2.putText(img, label, (x1, max(0, y1 - 10)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(img_path)[:30], fontsize=9)
        axes[i].axis('off')

    for j in range(len(samples), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Примеры из {split} ({len(images)} изображений)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{SIGNS_WORK}/check_{split}.png", dpi=150)
    plt.show()
    print(f"✅ Визуализация сохранена: {SIGNS_WORK}/check_{split}.png")

visualize_samples('train', 6)
visualize_samples('val', 3)

print("✅ Ячейка 6 готова.")

⚠️  Нет изображений в train
⚠️  Нет изображений в val
✅ Ячейка 6 готова.


In [ ]:
  ##############################################################
  # ЯЧЕЙКА 1: Установка
  ##############################################################
  !pip -q install ultralytics opencv-python-headless

  import os, shutil, time, json, glob, random, zipfile
  import numpy as np
  import cv2
  from collections import Counter
  from scipy.optimize import linear_sum_assignment
  from tqdm import tqdm
  from ultralytics import YOLO
  from google.colab import files, drive
  import matplotlib
  matplotlib.use('Agg')
  import matplotlib.pyplot as plt

  print("✅ Ячейка 1 готова.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ячейка 1 готова.


In [ ]:

##############################################################
# ЯЧЕЙКА 7: ОБУЧЕНИЕ YOLO НА ДОРОЖНЫХ ЗНАКАХ
##############################################################

# ═══════════════════════════════════════════════════
# НАСТРОЙКИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════
BASE_MODEL = "yolov8n.pt"       # можно yolov8s.pt для лучшего качества
EPOCHS     = 80                 # количество эпох (для ВКР 50-100)
BATCH_SIZE = 16                 # уменьшить до 8 если мало GPU-памяти
IMG_SIZE   = 640                # размер входного изображения
PATIENCE   = 15                 # ранняя остановка

# ═══════════════════════════════════════════════════
# ЗАГРУЗКА МОДЕЛИ И ЗАПУСК ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════
print(f"🚀 Обучение: {BASE_MODEL}")
print(f"   Эпох: {EPOCHS}")
print(f"   Batch: {BATCH_SIZE}")
print(f"   ImgSize: {IMG_SIZE}")
print(f"   Data: {data_yaml_path}")

model = YOLO(BASE_MODEL)

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    mosaic=1.0,
    flipud=0.0,         # не переворачиваем вертикально (знаки имеют ориентацию)
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5,           # небольшой поворот
    translate=0.1,
    scale=0.3,
    project=SIGNS_WORK,
    name='train_signs',
    exist_ok=True,
    verbose=True,
)

print("✅ Обучение завершено!")



🚀 Обучение: yolov8n.pt
   Эпох: 80
   Batch: 16
   ImgSize: 640
   Data: /content/drive/MyDrive/VKR_RoadSigns/dataset_yolo/data.yaml
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/VKR_RoadSigns/dataset_yolo/data.yaml, degrees=5, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

FileNotFoundError: [34m[1mtrain: [0mError loading data from /content/drive/MyDrive/VKR_RoadSigns/dataset_yolo/images/train
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [ ]:

##############################################################
# ЯЧЕЙКА 8: СОХРАНЕНИЕ ЛУЧШЕЙ МОДЕЛИ
##############################################################

# Путь к лучшей модели
best_model_path = f"{SIGNS_WORK}/train_signs/weights/best.pt"

if os.path.exists(best_model_path):
    # Копируем в папку моделей ВКР
    dst = f"{MODEL_DIR}/signs_best.pt"
    shutil.copy2(best_model_path, dst)
    print(f"✅ Лучшая модель скопирована: {dst}")
    print(f"   Размер: {os.path.getsize(dst) / 1e6:.1f} МБ")
else:
    print("⚠️  Файл best.pt не найден!")
    # Попробуем найти последнюю
    last_path = f"{SIGNS_WORK}/train_signs/weights/last.pt"
    if os.path.exists(last_path):
        dst = f"{MODEL_DIR}/signs_last.pt"
        shutil.copy2(last_path, dst)
        print(f"   Скопирован last.pt: {dst}")

⚠️  Файл best.pt не найден!


In [ ]:

##############################################################
# ЯЧЕЙКА 9: ОЦЕНКА МОДЕЛИ + МЕТРИКИ ДЛЯ ДИССЕРТАЦИИ
##############################################################

model_eval = YOLO(f"{MODEL_DIR}/signs_best.pt")

# Валидация
val_results = model_eval.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=SIGNS_WORK,
    name='val_signs',
    exist_ok=True,
)

print(f"\n{'═' * 60}")
print(f" МЕТРИКИ ДЛЯ ДИССЕРТАЦИИ — Модель дорожных знаков")
print(f"{'═' * 60}")

print(f"\n📊 ОБЩИЕ МЕТРИКИ:")
print(f"   mAP@0.5:       {val_results.box.map50:.4f}")
print(f"   mAP@0.5:0.95:  {val_results.box.map:.4f}")
print(f"   Precision:      {val_results.box.mp:.4f}")
print(f"   Recall:         {val_results.box.mr:.4f}")

print(f"\n📊 ПО КЛАССАМ:")
print(f"   {'Класс':<12} {'AP@0.5':<10} {'AP@0.5:0.95':<12} {'P':<8} {'R':<8}")
print(f"   {'─' * 50}")

class_names = ['5_19_1', '5_19_2', '1_22', '1_23']
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):
        ap50 = val_results.box.ap50[i]
        ap = val_results.box.ap[i]
        print(f"   {name:<12} {ap50:<10.4f} {ap:<12.4f}")

# Копируем графики обучения
train_dir = f"{SIGNS_WORK}/train_signs"
for fig_name in ['results.png', 'confusion_matrix.png',
                 'F1_curve.png', 'PR_curve.png',
                 'P_curve.png', 'R_curve.png']:
    src = os.path.join(train_dir, fig_name)
    if os.path.exists(src):
        shutil.copy2(src, f"{METRICS}/{fig_name}")

print(f"\n📈 Графики обучения скопированы в: {METRICS}/")
print("✅ Ячейка 9 готова.")


NameError: name 'YOLO' is not defined

In [ ]:

##############################################################
# ЯЧЕЙКА 10: ИНТЕГРАЦИЯ МОДЕЛИ ЗНАКОВ В ОСНОВНУЮ СИСТЕМУ
# ============================================================
# Обновление System класса для использования 3 моделей:
# 1. YOLOv8 COCO (person, car, bus, ...)
# 2. YOLOv8 Crosswalk (пешеходный переход)
# 3. YOLOv8 Signs (дорожные знаки)
##############################################################

print("""
╔══════════════════════════════════════════════════════════╗
║  ИНТЕГРАЦИЯ В ОСНОВНУЮ СИСТЕМУ                         ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Для использования модели знаков в основной системе      ║
║  добавьте в класс System:                                ║
║                                                          ║
║  1. Загрузку модели:                                     ║
║     self.m3 = YOLO("models/signs_best.pt")               ║
║                                                          ║
║  2. Детекцию в proc():                                   ║
║     r3 = self.m3(pf, conf=0.40, verbose=False)           ║
║     for b in r3[0].boxes:                                ║
║         cls_name = self.m3.names[int(b.cls[0])]          ║
║         if cls_name in ['5_19_1', '5_19_2']:             ║
║             # Знак пешеходного перехода                  ║
║             crossing = True; cross_timer = 0              ║
║         elif cls_name == '1_22':                         ║
║             # Предупреждающий знак перехода              ║
║             crossing = True; cross_timer = 0              ║
║         elif cls_name == '1_23':                         ║
║             # Знак "Дети"                                ║
║             crossing = True; cross_timer = 0              ║
║                                                          ║
║  Модель знаков ДОПОЛНЯЕТ детекцию пешеходных переходов:  ║
║  - Знаки видны раньше, чем разметка на дороге           ║
║  - Повышает дальность обнаружения зоны перехода          ║
║  - Знак "Дети" (1_23) → особая осторожность              ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

# Проверяем наличие всех моделей
print("📁 Проверка моделей:")
for name, path in [
    ("COCO (yolov8n.pt)",    f"{MODEL_DIR}/yolov8n.pt"),
    ("Crosswalk",            f"{MODEL_DIR}/crosswalk_best.pt"),
    ("Road Signs",           f"{MODEL_DIR}/signs_best.pt"),
]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"   ✅ {name}: {size:.1f} МБ")
    else:
        print(f"   ❌ {name}: не найдена ({path})")

print("\n✅ Всё готово для ВКР!")


╔══════════════════════════════════════════════════════════╗
║  ИНТЕГРАЦИЯ В ОСНОВНУЮ СИСТЕМУ                         ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Для использования модели знаков в основной системе      ║
║  добавьте в класс System:                                ║
║                                                          ║
║  1. Загрузку модели:                                     ║
║     self.m3 = YOLO("models/signs_best.pt")               ║
║                                                          ║
║  2. Детекцию в proc():                                   ║
║     r3 = self.m3(pf, conf=0.40, verbose=False)           ║
║     for b in r3[0].boxes:                                ║
║         cls_name = self.m3.names[int(b.cls[0])]          ║
║         if cls_name in ['5_19_1', '5_19_2']:             ║
║             # Знак пешеходного перехода                  ║
║             crossing = 

In [ ]:
import os
import glob

MODEL_DIR = "/content/drive/MyDrive/VKR_Crosswalk/models"

print("MODEL_DIR exists:", os.path.exists(MODEL_DIR))
print("Files in MODEL_DIR:")
if os.path.exists(MODEL_DIR):
    for f in os.listdir(MODEL_DIR):
        print("  ", f)

print("\nИщу все .pt модели в VKR_Crosswalk и VKR_RoadSigns:")

roots = [
    "/content/drive/MyDrive/VKR_Crosswalk",
    "/content/drive/MyDrive/VKR_RoadSigns"
]

pt_files = []
for root in roots:
    pt_files.extend(glob.glob(f"{root}/**/*.pt", recursive=True))

for p in pt_files:
    print(p)

MODEL_DIR exists: True
Files in MODEL_DIR:

Ищу все .pt модели в VKR_Crosswalk и VKR_RoadSigns:


In [ ]:
import os
import glob
import shutil

MODEL_DIR = "/content/drive/MyDrive/VKR_Crosswalk/models"
os.makedirs(MODEL_DIR, exist_ok=True)

best_files = glob.glob(
    "/content/drive/MyDrive/VKR_RoadSigns/runs/**/weights/best.pt",
    recursive=True
)

print("Найденные best.pt:")
for p in best_files:
    print(p)

if not best_files:
    raise FileNotFoundError("best.pt для Road Signs не найден.")

latest_best = max(best_files, key=os.path.getmtime)

dst = f"{MODEL_DIR}/signs_best.pt"
shutil.copy2(latest_best, dst)

print("✅ signs_best.pt скопирован:")
print(dst)

Найденные best.pt:


FileNotFoundError: best.pt для Road Signs не найден.

И СНОВА МЫ ЗДЕСЬ

In [ ]:
# ============================================================
# ПОЛНОСТЬЮ САМОСТОЯТЕЛЬНАЯ ЯЧЕЙКА
# Видео ВСЕГДА загружается вручную
# Используются 3 модели:
# 1) YOLOv8 COCO
# 2) Crosswalk
# 3) Road Signs
# После обработки результат автоматически скачивается
# ============================================================
!pip -q install ultralytics opencv-python-headless
import os
import time
import cv2
import numpy as np
from collections import Counter

from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import files

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. ПУТИ
# ============================================================

WORK = "/content/drive/MyDrive/VKR_Crosswalk"
MODEL_DIR = f"{WORK}/models"
RESULTS = f"{WORK}/results"
VIDEO_DIR = f"{WORK}/videos"

os.makedirs(WORK, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

coco_path = f"{MODEL_DIR}/yolov8n.pt"
cw_path = f"{MODEL_DIR}/crosswalk_best.pt"
signs_path = f"{MODEL_DIR}/signs_best.pt"


# ============================================================
# 2. ЗАГРУЗКА ВИДЕО ВРУЧНУЮ
# ============================================================

print("Загрузите видеофайл вручную:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Видео не было загружено.")

video_name = next(iter(uploaded))

if not video_name.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
    raise ValueError("Нужно загрузить видео в формате .mp4, .avi, .mov или .mkv")

VIDEO_PATH = f"{VIDEO_DIR}/{video_name}"

with open(VIDEO_PATH, "wb") as f:
    f.write(uploaded[video_name])

print(f"Видео успешно загружено: {VIDEO_PATH}")


# ============================================================
# 3. НАСТРОЙКИ
# ============================================================

PROC_W, PROC_H = 1280, 720
CONF_THRESHOLD = 0.30

V_AUTO = 50 / 3.6
MU = 0.7


# ============================================================
# 4. ПРОВЕРКА НАЛИЧИЯ МОДЕЛЕЙ
# ============================================================

if not os.path.exists(coco_path):
    print(f"COCO model not found: {coco_path}")
    print("Будет использована стандартная загрузка YOLO('yolov8n.pt').")
    coco_path = "yolov8n.pt"

if not os.path.exists(cw_path):
    print(f"Crosswalk model not found: {cw_path}")
    print("Модель пешеходного перехода не будет использована.")
    cw_path = None

if not os.path.exists(signs_path):
    print(f"Road Signs model not found: {signs_path}")
    print("Модель дорожных знаков не будет использована.")
    signs_path = None


# ============================================================
# 5. ЗАГРУЗКА МОДЕЛЕЙ
# ============================================================

print("\nLoading models...")

m_coco = YOLO(coco_path)
print("✅ YOLOv8 COCO loaded")

if cw_path is not None:
    m_cw = YOLO(cw_path)
    print("✅ Crosswalk model loaded")
    print(f"Crosswalk classes: {m_cw.names}")
else:
    m_cw = None

if signs_path is not None:
    m_signs = YOLO(signs_path)
    print("✅ Road Signs model loaded")
    print(f"Road Signs classes: {m_signs.names}")
else:
    m_signs = None

print("\n═══════════════════════════════════════════════════════")
print("СТАТУС МОДЕЛЕЙ ВКР")
print("═══════════════════════════════════════════════════════")
print("  ✅ YOLOv8 COCO")
print("  ✅ Crosswalk" if m_cw is not None else "  ❌ Crosswalk не загружена")
print("  ✅ Road Signs" if m_signs is not None else "  ❌ Road Signs не загружена")
print("═══════════════════════════════════════════════════════\n")


# ============================================================
# 6. БЫСТРАЯ ПРОВЕРКА МОДЕЛИ ПЕРЕХОДА НА ОДНОМ КАДРЕ
# ============================================================

if m_cw is not None:
    print("\n=== ПРОВЕРКА КАЧЕСТВА МОДЕЛИ ПЕРЕХОДА ===")

    cap_test = cv2.VideoCapture(VIDEO_PATH)
    cap_test.set(cv2.CAP_PROP_POS_FRAMES, 50)
    ok_test, frame_test = cap_test.read()
    cap_test.release()

    if ok_test:
        frame_small = cv2.resize(frame_test, (PROC_W, PROC_H))

        results_015 = m_cw(frame_small, conf=0.15, verbose=False)

        print("\nDetections at conf=0.15:")
        if len(results_015[0].boxes) == 0:
            print("   No detections")
        else:
            for b in results_015[0].boxes:
                cls = m_cw.names[int(b.cls[0])]
                conf = float(b.conf[0])
                print(f"   {cls}: {conf:.3f}")

        results_025 = m_cw(frame_small, conf=0.25, verbose=False)

        print("\nDetections at conf=0.25:")
        if len(results_025[0].boxes) == 0:
            print("   No detections")
        else:
            for b in results_025[0].boxes:
                cls = m_cw.names[int(b.cls[0])]
                conf = float(b.conf[0])
                print(f"   {cls}: {conf:.3f}")
    else:
        print("Не удалось прочитать 50-й кадр видео для теста.")


# ============================================================
# 7. КАЛМАНОВСКИЙ ФИЛЬТР
# ============================================================

class KF:
    def __init__(self):
        self.F = np.eye(8)

        for i in range(4):
            self.F[i, 4 + i] = 1.0

        self.H = np.eye(4, 8)
        self.sp = 1.0 / 20
        self.sv = 1.0 / 160

    def init(self, m):
        mn = np.zeros(8)
        mn[:4] = m

        st = (
            [2 * self.sp * m[3]] * 2
            + [1e-2, 2 * self.sp * m[3]]
            + [10 * self.sv * m[3]] * 2
            + [1e-5, 10 * self.sv * m[3]]
        )

        return mn, np.diag(np.square(st))

    def pred(self, mn, cv):
        st = (
            [self.sp * mn[3]] * 2
            + [1e-2, self.sp * mn[3]]
            + [self.sv * mn[3]] * 2
            + [1e-5, self.sv * mn[3]]
        )

        mn_new = self.F @ mn
        cv_new = self.F @ cv @ self.F.T + np.diag(np.square(st))

        return mn_new, cv_new

    def upd(self, mn, cv, z):
        st = [self.sp * mn[3]] * 2 + [1e-2, self.sp * mn[3]]
        R = np.diag(np.square(st))

        S = self.H @ cv @ self.H.T + R
        K = cv @ self.H.T @ np.linalg.inv(S)

        mn_new = mn + K @ (z - self.H @ mn)
        cv_new = (np.eye(8) - K @ self.H) @ cv

        return mn_new, cv_new


# ============================================================
# 8. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ ТРЕКИНГА
# ============================================================

def iou_(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(1, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(1, (b[2] - b[0]) * (b[3] - b[1]))

    union = area_a + area_b - inter

    return inter / union


def to_z(b):
    w = b[2] - b[0]
    h = max(b[3] - b[1], 1)

    cx = b[0] + w / 2
    cy = b[1] + h / 2
    aspect = w / h

    return np.array([cx, cy, aspect, h])


def to_b(z):
    w = z[2] * z[3]

    x1 = z[0] - w / 2
    y1 = z[1] - z[3] / 2
    x2 = z[0] + w / 2
    y2 = z[1] + z[3] / 2

    return [x1, y1, x2, y2]


# ============================================================
# 9. КЛАСС ТРЕКА
# ============================================================

class Trk:
    _n = 1

    def __init__(self, bb, cl, cf):
        self.id = Trk._n
        Trk._n += 1

        self.cls = cl
        self.cf = cf

        self.kf = KF()
        self.mn, self.cv = self.kf.init(to_z(bb))

        self.hits = 1
        self.age = 0

        self.tr = [(self.mn[0], self.mn[1])]
        self.bbs = [bb]

    def predict(self):
        self.mn, self.cv = self.kf.pred(self.mn, self.cv)
        self.age += 1

    def update(self, bb, cl, cf):
        self.mn, self.cv = self.kf.upd(self.mn, self.cv, to_z(bb))

        self.hits += 1
        self.age = 0
        self.cls = cl
        self.cf = cf

        self.tr.append((self.mn[0], self.mn[1]))
        self.bbs.append(bb)

        if len(self.tr) > 90:
            self.tr.pop(0)

        if len(self.bbs) > 90:
            self.bbs.pop(0)

    def bbox(self):
        return to_b(self.mn[:4])

    def ok(self):
        return self.hits >= 2


# ============================================================
# 10. ТРЕКЕР
# ============================================================

class Tracker:
    def __init__(self):
        self.tracks = []

    def update(self, dets):
        for t in self.tracks:
            t.predict()

        if self.tracks and dets:
            C = np.zeros((len(self.tracks), len(dets)))

            for i, t in enumerate(self.tracks):
                for j, d in enumerate(dets):
                    C[i, j] = 1 - iou_(t.bbox(), d["bbox"])

            rs, cs = linear_sum_assignment(C)

            matched_dets = set()

            for r, c in zip(rs, cs):
                if C[r, c] < 0.75:
                    self.tracks[r].update(
                        dets[c]["bbox"],
                        dets[c]["class"],
                        dets[c]["conf"]
                    )
                    matched_dets.add(c)

            for j, d in enumerate(dets):
                if j not in matched_dets:
                    self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        elif dets:
            for d in dets:
                self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        self.tracks = [t for t in self.tracks if t.age <= 45]

        return [t for t in self.tracks if t.ok()]


# ============================================================
# 11. ОЦЕНКА РАССТОЯНИЯ
# ============================================================

HEIGHTS = {
    "person": 1.70,
    "bicycle": 1.10,
    "car": 1.50,
    "bus": 3.00,
    "truck": 3.50
}


def est_dist(bb, cl, focal=800):
    if cl not in HEIGHTS:
        return None

    h = bb[3] - bb[1]

    if h <= 10:
        return None

    d = (HEIGHTS[cl] * focal) / h

    if 0.5 < d < 80:
        return d

    return None


# ============================================================
# 12. TTC И РЕШЕНИЯ
# ============================================================

def s_stop():
    return V_AUTO * 0.2 + V_AUTO ** 2 / (2 * MU * 9.81)


def compute_ttc(d, v_app=0):
    vr = V_AUTO + v_app

    if vr > 0.3:
        return d / vr

    return 999.0


def v_approach(trk):
    if len(trk.bbs) < 4:
        return 0.0

    h1 = trk.bbs[-1][3] - trk.bbs[-1][1]
    h0 = trk.bbs[-4][3] - trk.bbs[-4][1]

    if h0 <= 0:
        return 0.0

    return max(0, (h1 - h0) / max(h0, 1) * 3.0)


# ============================================================
# 13. COCO КЛАССЫ И ФИЛЬТР СВЕТОФОРА
# ============================================================

COCO_TGT = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic light"
}


def valid_tl(bb):
    w = bb[2] - bb[0]
    h = bb[3] - bb[1]

    if w < 8 or h < 15:
        return False

    ratio = h / max(w, 1)

    if ratio < 1.5 or ratio > 5:
        return False

    if bb[1] > PROC_H * 0.6:
        return False

    return True


# ============================================================
# 14. КЛАССЫ ДОРОЖНЫХ ЗНАКОВ
# ============================================================

# Эти имена можно адаптировать под имена классов твоей модели signs_best.pt.
# Я добавил несколько вариантов, чтобы код работал и с названиями из RTSD,
# и с более простыми названиями типа ped_crossing.

PEDESTRIAN_SIGN_CLASSES = {
    "ped_crossing",
    "ped_zebra_cross",
    "5_19_1",
    "5_19_2",
    "5.19.1",
    "5.19.2",
    "1_22",
    "1.22",
    "crosswalk_sign",
    "pedestrian_crossing"
}

CHILDREN_SIGN_CLASSES = {
    "1_23",
    "1.23",
    "children",
    "children_sign",
    "school_zone"
}

TRAFFIC_LIGHT_SIGN_CLASSES = {
    "red_light",
    "green_light",
    "traffic_light",
    "traffic light"
}


def is_pedestrian_related_sign(cls_name):
    return cls_name in PEDESTRIAN_SIGN_CLASSES or cls_name in CHILDREN_SIGN_CLASSES


def is_traffic_light_related(cls_name):
    return cls_name in TRAFFIC_LIGHT_SIGN_CLASSES


# ============================================================
# 15. ИНИЦИАЛИЗАЦИЯ ВИДЕО
# ============================================================

tracker = Tracker()

dc = Counter()
timing = []
decisions_log = []
frame_decisions = []

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Не удалось открыть видео: {VIDEO_PATH}")

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if total <= 0:
    cap.release()

    cap = cv2.VideoCapture(VIDEO_PATH)
    total = 0

    while True:
        ok, _ = cap.read()
        if not ok:
            break
        total += 1

    cap.release()
    cap = cv2.VideoCapture(VIDEO_PATH)

base_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
out_path = f"{RESULTS}/{base_name}_result.mp4"

wr = cv2.VideoWriter(
    out_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

if not wr.isOpened():
    raise RuntimeError("Не удалось создать выходной видеофайл.")

print(f"\nVideo: {W}x{H}, {fps:.0f} fps, {total} frames")
print(f"Output path: {out_path}")


# ============================================================
# 16. ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ ВИДЕО
# ============================================================

crossing = False
cross_timer = 0
regulated = False
cw_bb = None

for idx in tqdm(range(total), desc="Processing"):
    ok, frame = cap.read()

    if not ok:
        break

    oh, ow = frame.shape[:2]

    pf = cv2.resize(frame, (PROC_W, PROC_H))

    sx = ow / PROC_W
    sy = oh / PROC_H

    t0 = time.time()

    dets = []
    sign_dets = []

    # --------------------------------------------------------
    # COCO DETECTION
    # --------------------------------------------------------

    t1 = time.time()

    r1 = m_coco(pf, verbose=False, conf=0.32)

    tc1 = (time.time() - t1) * 1000

    tl_found = False

    for b in r1[0].boxes:
        ci = int(b.cls[0])

        if ci not in COCO_TGT:
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0])
        cf = float(b.conf[0])
        nm = COCO_TGT[ci]

        if cf < 0.32:
            continue

        if nm == "traffic light":
            if cf >= 0.40 and valid_tl([x1, y1, x2, y2]):
                tl_found = True
                regulated = True
                dc["traffic_light_coco"] += 1
            continue

        dets.append({
            "bbox": [x1, y1, x2, y2],
            "class": nm,
            "conf": cf
        })

        dc[nm] += 1

    # --------------------------------------------------------
    # CROSSWALK DETECTION
    # --------------------------------------------------------

    t2 = time.time()
    tc2 = 0

    if m_cw is not None:
        r2 = m_cw(pf, verbose=False, conf=0.25)

        tc2 = (time.time() - t2) * 1000

        for b in r2[0].boxes:
            cf = float(b.conf[0])

            if cf >= 0.25:
                x1, y1, x2, y2 = map(int, b.xyxy[0])

                crossing = True
                cross_timer = 0
                cw_bb = [x1, y1, x2, y2]

                dc["crosswalk"] += 1

    # --------------------------------------------------------
    # ROAD SIGNS DETECTION
    # --------------------------------------------------------

    t_signs = time.time()
    tc_signs = 0

    if m_signs is not None:
        r3 = m_signs(pf, verbose=False, conf=0.40)
        tc_signs = (time.time() - t_signs) * 1000

        for b in r3[0].boxes:
            cls_name = m_signs.names[int(b.cls[0])]
            cf = float(b.conf[0])
            x1, y1, x2, y2 = map(int, b.xyxy[0])

            sign_dets.append({
                "bbox": [x1, y1, x2, y2],
                "class": cls_name,
                "conf": cf
            })

            if is_pedestrian_related_sign(cls_name):
                crossing = True
                cross_timer = 0
                dc["sign_" + cls_name] += 1

            if is_traffic_light_related(cls_name):
                regulated = True
                crossing = True
                cross_timer = 0
                dc["sign_" + cls_name] += 1

    # --------------------------------------------------------
    # ОБНОВЛЕНИЕ СОСТОЯНИЯ ЗОНЫ ПЕРЕХОДА
    # --------------------------------------------------------

    if tl_found and not crossing:
        crossing = True
        cross_timer = 0

    if crossing:
        cross_timer += 1

        if cross_timer > 150:
            crossing = False
            regulated = False
            cw_bb = None

    # --------------------------------------------------------
    # TRACKING
    # --------------------------------------------------------

    t3 = time.time()

    active = tracker.update(dets)

    tc3 = (time.time() - t3) * 1000

    # --------------------------------------------------------
    # DECISION LOGIC
    # --------------------------------------------------------

    t4 = time.time()

    per_track = []
    overall = "NO_ACTION"
    reason = "Crossing zone not detected"

    if crossing:
        persons = []

        for t in active:
            if t.cls != "person":
                continue

            d = est_dist(t.bbox(), "person")

            if d is None:
                continue

            va = v_approach(t)
            tc_val = compute_ttc(d, va)

            persons.append((t, d, tc_val))

        ss = s_stop()

        if not persons:
            overall = "SLOW_DOWN"
            reason = "Crossing or pedestrian-related sign detected"

        else:
            worst = 0

            for t, d, tc_val in persons:
                if d <= 10 or tc_val < 1.5 or d <= ss:
                    a = "EMERGENCY_BRAKE"
                    p = 3

                elif d <= 20 or tc_val < 2.5:
                    a = "SLOW_DOWN"
                    p = 2

                elif d <= 35 or tc_val < 4.0:
                    a = "WARNING"
                    p = 1

                else:
                    a = "NO_ACTION"
                    p = 0

                per_track.append((t.id, a, tc_val, d))

                if p > worst:
                    worst = p
                    overall = a

            closest = min(persons, key=lambda x: x[1])

            prefix = "[Regulated]" if regulated else "[Unregulated]"

            labels = {
                "EMERGENCY_BRAKE": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - EMERGENCY",
                "SLOW_DOWN": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - slow down",
                "WARNING": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - warning",
                "NO_ACTION": f"{prefix} Pedestrians at safe distance"
            }

            reason = labels.get(overall, "")

    tc4 = (time.time() - t4) * 1000

    # --------------------------------------------------------
    # DRAWING
    # --------------------------------------------------------

    t5 = time.time()

    img = frame.copy()

    def sb(b):
        return [
            int(b[0] * sx),
            int(b[1] * sy),
            int(b[2] * sx),
            int(b[3] * sy)
        ]

    # Crosswalk zone
    if cw_bb is not None and crossing:
        cb = sb(cw_bb)

        cv2.rectangle(
            img,
            (cb[0], cb[1]),
            (cb[2], cb[3]),
            (0, 255, 0),
            4
        )

        cv2.putText(
            img,
            "CROSSWALK",
            (cb[0], max(0, cb[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 255, 0),
            3
        )

    # Road signs
    for sdet in sign_dets:
        b = sb(sdet["bbox"])
        cls_name = sdet["class"]
        cf = sdet["conf"]

        if is_pedestrian_related_sign(cls_name):
            sign_color = (255, 200, 0)
        elif is_traffic_light_related(cls_name):
            sign_color = (0, 255, 255)
        else:
            sign_color = (180, 180, 180)

        cv2.rectangle(
            img,
            (b[0], b[1]),
            (b[2], b[3]),
            sign_color,
            3
        )

        cv2.putText(
            img,
            f"SIGN:{cls_name} {cf:.2f}",
            (b[0], max(0, b[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            sign_color,
            2
        )

    # Detections
    for d in dets:
        if d["conf"] < CONF_THRESHOLD:
            continue

        b = sb(d["bbox"])

        if d["class"] == "person":
            color = (0, 0, 255)
        else:
            color = (255, 165, 0)

        cv2.rectangle(
            img,
            (b[0], b[1]),
            (b[2], b[3]),
            color,
            3
        )

        lbl = f"{d['class']} {d['conf']:.2f}"

        dst = est_dist(d["bbox"], d["class"])

        if dst:
            lbl += f" {dst:.1f}m"

        cv2.putText(
            img,
            lbl,
            (b[0], max(0, b[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            color,
            2
        )

    # Tracks
    for t in active:
        if len(t.tr) > 2:
            pts = np.array(
                [(int(p[0] * sx), int(p[1] * sy)) for p in t.tr],
                np.int32
            )

            cv2.polylines(
                img,
                [pts],
                False,
                (0, 255, 255),
                4
            )

            lp = pts[-1]

            cv2.putText(
                img,
                f"ID:{t.id}",
                (lp[0] - 20, lp[1] - 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 255),
                3
            )

    # Top panel
    ov = img.copy()

    cv2.rectangle(
        ov,
        (0, 0),
        (ow, 220),
        (0, 0, 0),
        -1
    )

    cv2.addWeighted(
        ov,
        0.7,
        img,
        0.3,
        0,
        img
    )

    # Crossing type
    if crossing:
        ct = "REGULATED" if regulated else "UNREGULATED"
        cc = (0, 200, 255) if regulated else (0, 100, 255)
    else:
        ct = "NOT DETECTED"
        cc = (0, 200, 0)

    cv2.putText(
        img,
        f"CROSSING: {ct}",
        (20, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.3,
        cc,
        3
    )

    # Decision
    acm = {
        "EMERGENCY_BRAKE": ((0, 0, 255), "EMERGENCY BRAKE"),
        "SLOW_DOWN": ((0, 165, 255), "SLOW DOWN"),
        "WARNING": ((0, 255, 255), "WARNING"),
        "NO_ACTION": ((0, 200, 0), "NO ACTION")
    }

    ac, al = acm.get(overall, ((255, 255, 255), overall))

    cv2.putText(
        img,
        f"DECISION: {al}",
        (20, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.6,
        ac,
        4
    )

    cv2.putText(
        img,
        reason[:80],
        (20, 145),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (200, 200, 200),
        2
    )

    ss_val = s_stop()
    np_ = len([t for t in active if t.cls == "person"])

    cv2.putText(
        img,
        f"v={V_AUTO * 3.6:.0f}km/h  S_stop={ss_val:.1f}m  Pedestrians={np_}",
        (20, 185),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (180, 180, 180),
        2
    )

    # Per-pedestrian table
    xr = max(20, ow - 700)

    cv2.putText(
        img,
        "PEDESTRIANS:",
        (xr, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2
    )

    for i, (tid, a2, tc_v, d2) in enumerate(per_track[:5]):
        yy = 80 + i * 32

        c2, a2l = acm.get(a2, ((255, 255, 255), "?"))

        cv2.putText(
            img,
            f"ID:{tid} D={d2:.1f}m TTC={min(tc_v, 99):.1f}s {a2l}",
            (xr, yy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            c2,
            2
        )

    tc5 = (time.time() - t5) * 1000
    tt = (time.time() - t0) * 1000

    # Logs
    timing.append({
        "c1": tc1,
        "c2": tc2,
        "signs": tc_signs,
        "tr": tc3,
        "dm": tc4,
        "vis": tc5,
        "tot": tt
    })

    frame_decisions.append((idx, overall, reason))

    for item in per_track:
        decisions_log.append({
            "f": idx,
            "id": item[0],
            "act": item[1],
            "ttc": item[2],
            "dist": item[3]
        })

    wr.write(img)


cap.release()
wr.release()


# ============================================================
# 17. МЕТРИКИ И РЕЗУЛЬТАТЫ
# ============================================================

print(f"\n{'=' * 60}")
print("RESULTS")
print("=" * 60)

print("\nDetections:")

if len(dc) == 0:
    print("   No detections")
else:
    for c, n in dc.most_common():
        print(f"   {c:<25}: {n}")

if timing:
    print("\nTiming per frame:")

    for k, nm in [
        ("c1", "COCO det"),
        ("c2", "Crosswalk det"),
        ("signs", "Road signs det"),
        ("tr", "Tracking"),
        ("dm", "Decision"),
        ("vis", "Drawing"),
        ("tot", "TOTAL")
    ]:
        vals = [t[k] for t in timing]
        print(f"   {nm:<20}: {np.mean(vals):.1f} ms")

    avg_total = np.mean([t["tot"] for t in timing])

    if avg_total > 0:
        print(f"   FPS: {1000 / avg_total:.1f}")

if frame_decisions:
    print("\nDecisions summary:")

    acounter = Counter(f[1] for f in frame_decisions)
    tf = len(frame_decisions)

    for a, n in acounter.most_common():
        label = {
            "EMERGENCY_BRAKE": "EMERGENCY BRAKE",
            "SLOW_DOWN": "SLOW DOWN",
            "WARNING": "WARNING",
            "NO_ACTION": "NO ACTION"
        }.get(a, a)

        print(f"   {label:<25}: {n} frames ({n / tf * 100:.1f}%)")

print("\nBraking distance, v=50 km/h:")

for nm, mu in [
    ("Dry", 0.7),
    ("Wet", 0.5),
    ("Ice", 0.3)
]:
    v = V_AUTO

    driver_stop = v * 1.2 + v ** 2 / (2 * mu * 9.81)
    system_stop = v * 0.2 + v ** 2 / (2 * mu * 9.81)
    gain = driver_stop - system_stop

    print(
        f"   {nm}: "
        f"driver={driver_stop:.1f}m, "
        f"system={system_stop:.1f}m, "
        f"gain={gain:.1f}m"
    )


# ============================================================
# 18. АВТОМАТИЧЕСКОЕ СКАЧИВАНИЕ РЕЗУЛЬТАТА
# ============================================================

print(f"\nDone! Video saved to: {out_path}")

if os.path.exists(out_path):
    print("Начинается автоматическое скачивание видео...")
    time.sleep(1)
    files.download(out_path)
else:
    print(f"Файл результата не найден: {out_path}")

Загрузите видеофайл вручную:


MessageError: CustomError: Timed out waiting for iframe configuration. URL: https://colab.research.google.com/drive/1a15jolKHje1CIgdYBl70cQQnD-tmJOsz#scrollTo=HqyjgekwOKr2

окончательная проверка

In [ ]:
# ============================================================
# ПОЛНОСТЬЮ САМОСТОЯТЕЛЬНАЯ ЯЧЕЙКА
# Можно загрузить сразу 9 видео
# Все обработанные видео сохраняются, затем автоматически скачиваются одним ZIP-архивом
#
# Используются 3 модели:
# 1) YOLOv8 COCO
# 2) Crosswalk
# 3) Road Signs
# ============================================================

!pip -q install ultralytics opencv-python-headless

import os
import time
import cv2
import zipfile
import numpy as np
from collections import Counter
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import files

import matplotlib
matplotlib.use("Agg")


# ============================================================
# 1. ПУТИ
# ============================================================

WORK = "/content/drive/MyDrive/VKR_Crosswalk"
MODEL_DIR = f"{WORK}/models"
RESULTS = f"{WORK}/results"
VIDEO_DIR = f"{WORK}/videos"

os.makedirs(WORK, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

coco_path = f"{MODEL_DIR}/yolov8n.pt"
cw_path = f"{MODEL_DIR}/crosswalk_best.pt"
signs_path = f"{MODEL_DIR}/signs_best.pt"


# ============================================================
# 2. ЗАГРУЗКА НЕСКОЛЬКИХ ВИДЕО ВРУЧНУЮ
# ============================================================

print("Загрузите сразу до 9 видеофайлов:")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Видео не были загружены.")

VIDEO_PATHS = []

for video_name, video_bytes in uploaded.items():
    if not video_name.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
        print(f"Пропущен файл не-видео: {video_name}")
        continue

    video_path = f"{VIDEO_DIR}/{video_name}"

    with open(video_path, "wb") as f:
        f.write(video_bytes)

    VIDEO_PATHS.append(video_path)
    print(f"Видео загружено: {video_path}")

if len(VIDEO_PATHS) == 0:
    raise RuntimeError("Не найдено ни одного видеофайла.")

if len(VIDEO_PATHS) > 9:
    print(f"Загружено {len(VIDEO_PATHS)} видео. Будут обработаны только первые 9.")
    VIDEO_PATHS = VIDEO_PATHS[:9]

print(f"\nВсего видео для обработки: {len(VIDEO_PATHS)}")


# ============================================================
# 3. НАСТРОЙКИ
# ============================================================

PROC_W, PROC_H = 1280, 720
CONF_THRESHOLD = 0.30

V_AUTO = 50 / 3.6
MU = 0.7


# ============================================================
# 4. ПРОВЕРКА НАЛИЧИЯ МОДЕЛЕЙ
# ============================================================

if not os.path.exists(coco_path):
    print(f"COCO model not found: {coco_path}")
    print("Будет использована стандартная загрузка YOLO('yolov8n.pt').")
    coco_path = "yolov8n.pt"

if not os.path.exists(cw_path):
    print(f"Crosswalk model not found: {cw_path}")
    print("Модель пешеходного перехода не будет использована.")
    cw_path = None

if not os.path.exists(signs_path):
    print(f"Road Signs model not found: {signs_path}")
    print("Модель дорожных знаков не будет использована.")
    signs_path = None


# ============================================================
# 5. ЗАГРУЗКА МОДЕЛЕЙ
# ============================================================

print("\nLoading models...")

m_coco = YOLO(coco_path)
print("✅ YOLOv8 COCO loaded")

if cw_path is not None:
    m_cw = YOLO(cw_path)
    print("✅ Crosswalk model loaded")
    print(f"Crosswalk classes: {m_cw.names}")
else:
    m_cw = None

if signs_path is not None:
    m_signs = YOLO(signs_path)
    print("✅ Road Signs model loaded")
    print(f"Road Signs classes: {m_signs.names}")
else:
    m_signs = None

print("\n═══════════════════════════════════════════════════════")
print("СТАТУС МОДЕЛЕЙ ВКР")
print("═══════════════════════════════════════════════════════")
print("  ✅ YOLOv8 COCO")
print("  ✅ Crosswalk" if m_cw is not None else "  ❌ Crosswalk не загружена")
print("  ✅ Road Signs" if m_signs is not None else "  ❌ Road Signs не загружена")
print("═══════════════════════════════════════════════════════\n")


# ============================================================
# 6. КАЛМАНОВСКИЙ ФИЛЬТР
# ============================================================

class KF:
    def __init__(self):
        self.F = np.eye(8)

        for i in range(4):
            self.F[i, 4 + i] = 1.0

        self.H = np.eye(4, 8)
        self.sp = 1.0 / 20
        self.sv = 1.0 / 160

    def init(self, m):
        mn = np.zeros(8)
        mn[:4] = m

        st = (
            [2 * self.sp * m[3]] * 2
            + [1e-2, 2 * self.sp * m[3]]
            + [10 * self.sv * m[3]] * 2
            + [1e-5, 10 * self.sv * m[3]]
        )

        return mn, np.diag(np.square(st))

    def pred(self, mn, cv):
        st = (
            [self.sp * mn[3]] * 2
            + [1e-2, self.sp * mn[3]]
            + [self.sv * mn[3]] * 2
            + [1e-5, self.sv * mn[3]]
        )

        mn_new = self.F @ mn
        cv_new = self.F @ cv @ self.F.T + np.diag(np.square(st))

        return mn_new, cv_new

    def upd(self, mn, cv, z):
        st = [self.sp * mn[3]] * 2 + [1e-2, self.sp * mn[3]]
        R = np.diag(np.square(st))

        S = self.H @ cv @ self.H.T + R
        K = cv @ self.H.T @ np.linalg.inv(S)

        mn_new = mn + K @ (z - self.H @ mn)
        cv_new = (np.eye(8) - K @ self.H) @ cv

        return mn_new, cv_new


# ============================================================
# 7. ТРЕКИНГ
# ============================================================

def iou_(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(1, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(1, (b[2] - b[0]) * (b[3] - b[1]))

    union = area_a + area_b - inter

    return inter / union


def to_z(b):
    w = b[2] - b[0]
    h = max(b[3] - b[1], 1)

    cx = b[0] + w / 2
    cy = b[1] + h / 2
    aspect = w / h

    return np.array([cx, cy, aspect, h])


def to_b(z):
    w = z[2] * z[3]

    x1 = z[0] - w / 2
    y1 = z[1] - z[3] / 2
    x2 = z[0] + w / 2
    y2 = z[1] + z[3] / 2

    return [x1, y1, x2, y2]


class Trk:
    _n = 1

    def __init__(self, bb, cl, cf):
        self.id = Trk._n
        Trk._n += 1

        self.cls = cl
        self.cf = cf

        self.kf = KF()
        self.mn, self.cv = self.kf.init(to_z(bb))

        self.hits = 1
        self.age = 0

        self.tr = [(self.mn[0], self.mn[1])]
        self.bbs = [bb]

    def predict(self):
        self.mn, self.cv = self.kf.pred(self.mn, self.cv)
        self.age += 1

    def update(self, bb, cl, cf):
        self.mn, self.cv = self.kf.upd(self.mn, self.cv, to_z(bb))

        self.hits += 1
        self.age = 0
        self.cls = cl
        self.cf = cf

        self.tr.append((self.mn[0], self.mn[1]))
        self.bbs.append(bb)

        if len(self.tr) > 90:
            self.tr.pop(0)

        if len(self.bbs) > 90:
            self.bbs.pop(0)

    def bbox(self):
        return to_b(self.mn[:4])

    def ok(self):
        return self.hits >= 2


class Tracker:
    def __init__(self):
        self.tracks = []

    def update(self, dets):
        for t in self.tracks:
            t.predict()

        if self.tracks and dets:
            C = np.zeros((len(self.tracks), len(dets)))

            for i, t in enumerate(self.tracks):
                for j, d in enumerate(dets):
                    C[i, j] = 1 - iou_(t.bbox(), d["bbox"])

            rs, cs = linear_sum_assignment(C)

            matched_dets = set()

            for r, c in zip(rs, cs):
                if C[r, c] < 0.75:
                    self.tracks[r].update(
                        dets[c]["bbox"],
                        dets[c]["class"],
                        dets[c]["conf"]
                    )
                    matched_dets.add(c)

            for j, d in enumerate(dets):
                if j not in matched_dets:
                    self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        elif dets:
            for d in dets:
                self.tracks.append(Trk(d["bbox"], d["class"], d["conf"]))

        self.tracks = [t for t in self.tracks if t.age <= 45]

        return [t for t in self.tracks if t.ok()]


# ============================================================
# 8. РАССТОЯНИЕ, TTC И РЕШЕНИЯ
# ============================================================

HEIGHTS = {
    "person": 1.70,
    "bicycle": 1.10,
    "car": 1.50,
    "bus": 3.00,
    "truck": 3.50
}


def est_dist(bb, cl, focal=800):
    if cl not in HEIGHTS:
        return None

    h = bb[3] - bb[1]

    if h <= 10:
        return None

    d = (HEIGHTS[cl] * focal) / h

    if 0.5 < d < 80:
        return d

    return None


def s_stop():
    return V_AUTO * 0.2 + V_AUTO ** 2 / (2 * MU * 9.81)


def compute_ttc(d, v_app=0):
    vr = V_AUTO + v_app

    if vr > 0.3:
        return d / vr

    return 999.0


def v_approach(trk):
    if len(trk.bbs) < 4:
        return 0.0

    h1 = trk.bbs[-1][3] - trk.bbs[-1][1]
    h0 = trk.bbs[-4][3] - trk.bbs[-4][1]

    if h0 <= 0:
        return 0.0

    return max(0, (h1 - h0) / max(h0, 1) * 3.0)


# ============================================================
# 9. КЛАССЫ
# ============================================================

COCO_TGT = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic light"
}


def valid_tl(bb):
    w = bb[2] - bb[0]
    h = bb[3] - bb[1]

    if w < 8 or h < 15:
        return False

    ratio = h / max(w, 1)

    if ratio < 1.5 or ratio > 5:
        return False

    if bb[1] > PROC_H * 0.6:
        return False

    return True


PEDESTRIAN_SIGN_CLASSES = {
    "ped_crossing",
    "ped_zebra_cross",
    "5_19_1",
    "5_19_2",
    "5.19.1",
    "5.19.2",
    "1_22",
    "1.22",
    "crosswalk_sign",
    "pedestrian_crossing"
}

CHILDREN_SIGN_CLASSES = {
    "1_23",
    "1.23",
    "children",
    "children_sign",
    "school_zone"
}

TRAFFIC_LIGHT_SIGN_CLASSES = {
    "red_light",
    "green_light",
    "traffic_light",
    "traffic light"
}


def is_pedestrian_related_sign(cls_name):
    return cls_name in PEDESTRIAN_SIGN_CLASSES or cls_name in CHILDREN_SIGN_CLASSES


def is_traffic_light_related(cls_name):
    return cls_name in TRAFFIC_LIGHT_SIGN_CLASSES


# ============================================================
# 10. ОБРАБОТКА ОДНОГО ВИДЕО
# ============================================================

def process_video(VIDEO_PATH):
    print("\n" + "=" * 80)
    print(f"ОБРАБОТКА ВИДЕО: {os.path.basename(VIDEO_PATH)}")
    print("=" * 80)

    tracker = Tracker()

    dc = Counter()
    timing = []
    decisions_log = []
    frame_decisions = []

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print(f"Не удалось открыть видео: {VIDEO_PATH}")
        return None

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()

        cap = cv2.VideoCapture(VIDEO_PATH)
        total = 0

        while True:
            ok, _ = cap.read()
            if not ok:
                break
            total += 1

        cap.release()
        cap = cv2.VideoCapture(VIDEO_PATH)

    base_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    out_path = f"{RESULTS}/{base_name}_result.mp4"

    wr = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (W, H)
    )

    if not wr.isOpened():
        cap.release()
        print(f"Не удалось создать выходной видеофайл: {out_path}")
        return None

    print(f"Video: {W}x{H}, {fps:.0f} fps, {total} frames")
    print(f"Output path: {out_path}")

    crossing = False
    cross_timer = 0
    regulated = False
    cw_bb = None

    for idx in tqdm(range(total), desc=f"Processing {base_name}"):
        ok, frame = cap.read()

        if not ok:
            break

        oh, ow = frame.shape[:2]

        pf = cv2.resize(frame, (PROC_W, PROC_H))

        sx = ow / PROC_W
        sy = oh / PROC_H

        t0 = time.time()

        dets = []
        sign_dets = []

        # --------------------------------------------------------
        # COCO DETECTION
        # --------------------------------------------------------

        t1 = time.time()

        r1 = m_coco(pf, verbose=False, conf=0.32)

        tc1 = (time.time() - t1) * 1000

        tl_found = False

        for b in r1[0].boxes:
            ci = int(b.cls[0])

            if ci not in COCO_TGT:
                continue

            x1, y1, x2, y2 = map(int, b.xyxy[0])
            cf = float(b.conf[0])
            nm = COCO_TGT[ci]

            if cf < 0.32:
                continue

            if nm == "traffic light":
                if cf >= 0.40 and valid_tl([x1, y1, x2, y2]):
                    tl_found = True
                    regulated = True
                    dc["traffic_light_coco"] += 1
                continue

            dets.append({
                "bbox": [x1, y1, x2, y2],
                "class": nm,
                "conf": cf
            })

            dc[nm] += 1

        # --------------------------------------------------------
        # CROSSWALK DETECTION
        # --------------------------------------------------------

        t2 = time.time()
        tc2 = 0

        if m_cw is not None:
            r2 = m_cw(pf, verbose=False, conf=0.25)

            tc2 = (time.time() - t2) * 1000

            for b in r2[0].boxes:
                cf = float(b.conf[0])

                if cf >= 0.25:
                    x1, y1, x2, y2 = map(int, b.xyxy[0])

                    crossing = True
                    cross_timer = 0
                    cw_bb = [x1, y1, x2, y2]

                    dc["crosswalk"] += 1

        # --------------------------------------------------------
        # ROAD SIGNS DETECTION
        # --------------------------------------------------------

        t_signs = time.time()
        tc_signs = 0

        if m_signs is not None:
            r3 = m_signs(pf, verbose=False, conf=0.40)
            tc_signs = (time.time() - t_signs) * 1000

            for b in r3[0].boxes:
                cls_name = m_signs.names[int(b.cls[0])]
                cf = float(b.conf[0])
                x1, y1, x2, y2 = map(int, b.xyxy[0])

                sign_dets.append({
                    "bbox": [x1, y1, x2, y2],
                    "class": cls_name,
                    "conf": cf
                })

                if is_pedestrian_related_sign(cls_name):
                    crossing = True
                    cross_timer = 0
                    dc["sign_" + cls_name] += 1

                if is_traffic_light_related(cls_name):
                    regulated = True
                    crossing = True
                    cross_timer = 0
                    dc["sign_" + cls_name] += 1

        # --------------------------------------------------------
        # ОБНОВЛЕНИЕ ЗОНЫ ПЕРЕХОДА
        # --------------------------------------------------------

        if tl_found and not crossing:
            crossing = True
            cross_timer = 0

        if crossing:
            cross_timer += 1

            if cross_timer > 150:
                crossing = False
                regulated = False
                cw_bb = None

        # --------------------------------------------------------
        # TRACKING
        # --------------------------------------------------------

        t3 = time.time()

        active = tracker.update(dets)

        tc3 = (time.time() - t3) * 1000

        # --------------------------------------------------------
        # DECISION LOGIC
        # --------------------------------------------------------

        t4 = time.time()

        per_track = []
        overall = "NO_ACTION"
        reason = "Crossing zone not detected"

        if crossing:
            persons = []

            for t in active:
                if t.cls != "person":
                    continue

                d = est_dist(t.bbox(), "person")

                if d is None:
                    continue

                va = v_approach(t)
                tc_val = compute_ttc(d, va)

                persons.append((t, d, tc_val))

            ss = s_stop()

            if not persons:
                overall = "SLOW_DOWN"
                reason = "Crossing or pedestrian-related sign detected"

            else:
                worst = 0

                for t, d, tc_val in persons:
                    if d <= 10 or tc_val < 1.5 or d <= ss:
                        a = "EMERGENCY_BRAKE"
                        p = 3

                    elif d <= 20 or tc_val < 2.5:
                        a = "SLOW_DOWN"
                        p = 2

                    elif d <= 35 or tc_val < 4.0:
                        a = "WARNING"
                        p = 1

                    else:
                        a = "NO_ACTION"
                        p = 0

                    per_track.append((t.id, a, tc_val, d))

                    if p > worst:
                        worst = p
                        overall = a

                closest = min(persons, key=lambda x: x[1])

                prefix = "[Regulated]" if regulated else "[Unregulated]"

                labels = {
                    "EMERGENCY_BRAKE": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - EMERGENCY",
                    "SLOW_DOWN": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - slow down",
                    "WARNING": f"{prefix} Ped ID:{closest[0].id} at {closest[1]:.1f}m - warning",
                    "NO_ACTION": f"{prefix} Pedestrians at safe distance"
                }

                reason = labels.get(overall, "")

        tc4 = (time.time() - t4) * 1000

        # --------------------------------------------------------
        # DRAWING
        # --------------------------------------------------------

        t5 = time.time()

        img = frame.copy()

        def sb(b):
            return [
                int(b[0] * sx),
                int(b[1] * sy),
                int(b[2] * sx),
                int(b[3] * sy)
            ]

        # Crosswalk zone
        if cw_bb is not None and crossing:
            cb = sb(cw_bb)

            cv2.rectangle(
                img,
                (cb[0], cb[1]),
                (cb[2], cb[3]),
                (0, 255, 0),
                4
            )

            cv2.putText(
                img,
                "CROSSWALK",
                (cb[0], max(0, cb[1] - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.2,
                (0, 255, 0),
                3
            )

        # Road signs
        for sdet in sign_dets:
            b = sb(sdet["bbox"])
            cls_name = sdet["class"]
            cf = sdet["conf"]

            if is_pedestrian_related_sign(cls_name):
                sign_color = (255, 200, 0)
            elif is_traffic_light_related(cls_name):
                sign_color = (0, 255, 255)
            else:
                sign_color = (180, 180, 180)

            cv2.rectangle(
                img,
                (b[0], b[1]),
                (b[2], b[3]),
                sign_color,
                3
            )

            cv2.putText(
                img,
                f"SIGN:{cls_name} {cf:.2f}",
                (b[0], max(0, b[1] - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                sign_color,
                2
            )

        # Detections
        for d in dets:
            if d["conf"] < CONF_THRESHOLD:
                continue

            b = sb(d["bbox"])

            if d["class"] == "person":
                color = (0, 0, 255)
            else:
                color = (255, 165, 0)

            cv2.rectangle(
                img,
                (b[0], b[1]),
                (b[2], b[3]),
                color,
                3
            )

            lbl = f"{d['class']} {d['conf']:.2f}"

            dst = est_dist(d["bbox"], d["class"])

            if dst:
                lbl += f" {dst:.1f}m"

            cv2.putText(
                img,
                lbl,
                (b[0], max(0, b[1] - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                color,
                2
            )

        # Tracks
        for t in active:
            if len(t.tr) > 2:
                pts = np.array(
                    [(int(p[0] * sx), int(p[1] * sy)) for p in t.tr],
                    np.int32
                )

                cv2.polylines(
                    img,
                    [pts],
                    False,
                    (0, 255, 255),
                    4
                )

                lp = pts[-1]

                cv2.putText(
                    img,
                    f"ID:{t.id}",
                    (lp[0] - 20, lp[1] - 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.0,
                    (0, 255, 255),
                    3
                )

        # Top panel
        ov = img.copy()

        cv2.rectangle(
            ov,
            (0, 0),
            (ow, 220),
            (0, 0, 0),
            -1
        )

        cv2.addWeighted(
            ov,
            0.7,
            img,
            0.3,
            0,
            img
        )

        if crossing:
            ct = "REGULATED" if regulated else "UNREGULATED"
            cc = (0, 200, 255) if regulated else (0, 100, 255)
        else:
            ct = "NOT DETECTED"
            cc = (0, 200, 0)

        cv2.putText(
            img,
            f"CROSSING: {ct}",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.3,
            cc,
            3
        )

        acm = {
            "EMERGENCY_BRAKE": ((0, 0, 255), "EMERGENCY BRAKE"),
            "SLOW_DOWN": ((0, 165, 255), "SLOW DOWN"),
            "WARNING": ((0, 255, 255), "WARNING"),
            "NO_ACTION": ((0, 200, 0), "NO ACTION")
        }

        ac, al = acm.get(overall, ((255, 255, 255), overall))

        cv2.putText(
            img,
            f"DECISION: {al}",
            (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.6,
            ac,
            4
        )

        cv2.putText(
            img,
            reason[:80],
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (200, 200, 200),
            2
        )

        ss_val = s_stop()
        np_ = len([t for t in active if t.cls == "person"])

        cv2.putText(
            img,
            f"v={V_AUTO * 3.6:.0f}km/h  S_stop={ss_val:.1f}m  Pedestrians={np_}",
            (20, 185),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (180, 180, 180),
            2
        )

        xr = max(20, ow - 700)

        cv2.putText(
            img,
            "PEDESTRIANS:",
            (xr, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2
        )

        for i, (tid, a2, tc_v, d2) in enumerate(per_track[:5]):
            yy = 80 + i * 32

            c2, a2l = acm.get(a2, ((255, 255, 255), "?"))

            cv2.putText(
                img,
                f"ID:{tid} D={d2:.1f}m TTC={min(tc_v, 99):.1f}s {a2l}",
                (xr, yy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                c2,
                2
            )

        tc5 = (time.time() - t5) * 1000
        tt = (time.time() - t0) * 1000

        timing.append({
            "c1": tc1,
            "c2": tc2,
            "signs": tc_signs,
            "tr": tc3,
            "dm": tc4,
            "vis": tc5,
            "tot": tt
        })

        frame_decisions.append((idx, overall, reason))

        for item in per_track:
            decisions_log.append({
                "f": idx,
                "id": item[0],
                "act": item[1],
                "ttc": item[2],
                "dist": item[3]
            })

        wr.write(img)

    cap.release()
    wr.release()

    # ========================================================
    # МЕТРИКИ ПО ОДНОМУ ВИДЕО
    # ========================================================

    print(f"\nRESULTS FOR: {os.path.basename(VIDEO_PATH)}")

    print("\nDetections:")
    if len(dc) == 0:
        print("   No detections")
    else:
        for c, n in dc.most_common():
            print(f"   {c:<25}: {n}")

    if timing:
        print("\nTiming per frame:")

        for k, nm in [
            ("c1", "COCO det"),
            ("c2", "Crosswalk det"),
            ("signs", "Road signs det"),
            ("tr", "Tracking"),
            ("dm", "Decision"),
            ("vis", "Drawing"),
            ("tot", "TOTAL")
        ]:
            vals = [t[k] for t in timing]
            print(f"   {nm:<20}: {np.mean(vals):.1f} ms")

        avg_total = np.mean([t["tot"] for t in timing])

        if avg_total > 0:
            print(f"   FPS: {1000 / avg_total:.1f}")

    if frame_decisions:
        print("\nDecisions summary:")

        acounter = Counter(f[1] for f in frame_decisions)
        tf = len(frame_decisions)

        for a, n in acounter.most_common():
            label = {
                "EMERGENCY_BRAKE": "EMERGENCY BRAKE",
                "SLOW_DOWN": "SLOW DOWN",
                "WARNING": "WARNING",
                "NO_ACTION": "NO ACTION"
            }.get(a, a)

            print(f"   {label:<25}: {n} frames ({n / tf * 100:.1f}%)")

    print(f"\nDone video: {out_path}")

    return out_path


# ============================================================
# 11. ОБРАБОТКА ВСЕХ ВИДЕО
# ============================================================

output_videos = []

for video_path in VIDEO_PATHS:
    out = process_video(video_path)
    if out is not None and os.path.exists(out):
        output_videos.append(out)

if len(output_videos) == 0:
    raise RuntimeError("Не было создано ни одного результата.")


# ============================================================
# 12. ОБЩИЙ ОТЧЁТ ПО ТОРМОЗНОМУ ПУТИ
# ============================================================

print("\n" + "=" * 80)
print("BRaking distance, v=50 km/h:")
print("=" * 80)

for nm, mu in [
    ("Dry", 0.7),
    ("Wet", 0.5),
    ("Ice", 0.3)
]:
    v = V_AUTO

    driver_stop = v * 1.2 + v ** 2 / (2 * mu * 9.81)
    system_stop = v * 0.2 + v ** 2 / (2 * mu * 9.81)
    gain = driver_stop - system_stop

    print(
        f"   {nm}: "
        f"driver={driver_stop:.1f}m, "
        f"system={system_stop:.1f}m, "
        f"gain={gain:.1f}m"
    )


# ============================================================
# 13. СОЗДАНИЕ ZIP С РЕЗУЛЬТАТАМИ И АВТОСКАЧИВАНИЕ
# ============================================================

zip_path = f"{RESULTS}/processed_9_videos_results.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

print("\nСоздаю ZIP-архив со всеми обработанными видео...")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for out_video in output_videos:
        z.write(out_video, arcname=os.path.basename(out_video))
        print(f"Добавлено в ZIP: {os.path.basename(out_video)}")

print(f"\nГотово. ZIP сохранён:")
print(zip_path)

print("\nНачинается автоматическое скачивание ZIP-архива...")
time.sleep(1)
files.download(zip_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Загрузите сразу до 9 видеофайлов:


Saving yolo_video_24fps.mp4 to yolo_video_24fps.mp4
Видео загружено: /content/drive/MyDrive/VKR_Crosswalk/videos/yolo_video_24fps.mp4

Всего видео для обработки: 1
COCO model not found: /content/drive/MyDrive/VKR_Crosswalk/models/yolov8n.pt
Будет использована стандартная загрузка YOLO('yolov8n.pt').
Crosswalk model not found: /content/drive/MyDrive/VKR_Crosswalk/models/crosswalk_best.pt
Модель пешеходного перехода не будет использована.
Road Signs model not found: /content/drive/MyDrive/VKR_Crosswalk/models/signs_best.pt
Модель дорожных знаков не будет использована.

Loading models...
✅ YOLOv8 COCO loaded

═══════════════════════════════════════════════════════
СТАТУС МОДЕЛЕЙ ВКР
═══════════════════════════════════════════════════════
  ✅ YOLOv8 COCO
  ❌ Crosswalk не загружена
  ❌ Road Signs не загружена
═══════════════════════════════════════════════════════


ОБРАБОТКА ВИДЕО: yolo_video_24fps.mp4
Video: 3840x2160, 24 fps, 174 frames
Output path: /content/drive/MyDrive/VKR_Crosswalk/r

Processing yolo_video_24fps:   2%|▏         | 4/174 [00:02<01:43,  1.65it/s]


KeyboardInterrupt: 